# 📊 RAG Evaluation Master Notebook

> **Purpose:** A production-grade, end-to-end evaluation framework for Retrieval-Augmented Generation (RAG) systems.  
> One notebook covering every metric, every pipeline stage, and every cost dimension — ready to drop into any RAG project.

---

## 📋 Table of Contents

| # | Section | What you'll evaluate |
|---|---------|----------------------|
| 1 | [Setup & Test Corpus](#1-setup) | API config, synthetic RAG pipeline, shared utilities |
| 2 | [Retrieval Quality Metrics](#2-retrieval) | Recall@K, Precision@K, MRR, NDCG, Hit Rate |
| 3 | [Generation Quality Metrics](#3-generation) | Faithfulness, Answer Relevance, ROUGE, BERTScore (sim) |
| 4 | [End-to-End RAG Metrics](#4-e2e) | Context Recall, Context Precision, Answer Correctness |
| 5 | [LLM-as-Judge Evaluation](#5-llm-judge) | Pairwise eval, rubric scoring, G-Eval framework |
| 6 | [Hallucination Detection](#6-hallucination) | Claim extraction, NLI scoring, grounding check |
| 7 | [Latency & Performance Profiling](#7-latency) | Stage-by-stage profiling, P50/P95/P99, bottleneck analysis |
| 8 | [Cost Analysis & Optimization](#8-cost) | Token cost per query, pipeline cost breakdown, ROI model |
| 9 | [Regression Testing Framework](#9-regression) | Baseline tracking, alert thresholds, CI/CD integration |
| 10 | [Evaluation Dashboard](#10-dashboard) | Full visual scorecard, radar charts, trend analysis |
| 11 | [🚀 Production Harness](#11-harness) | Drop-in evaluator for your RAG system |

---
> **Requirements:** Anthropic API key (set in Section 1). All metrics except LLM-judge run offline.


## 1. Setup & Test Corpus <a id='1-setup'></a>

We build a realistic synthetic RAG pipeline with a knowledge base, retriever, and generator — so every metric can be computed end-to-end without needing your real system yet.

In [ ]:
import os, re, json, time, math, warnings, hashlib, collections
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import requests

# ── Dark theme ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":"#0d1117","axes.facecolor":"#161b22",
    "axes.edgecolor":"#30363d","axes.labelcolor":"#c9d1d9",
    "xtick.color":"#8b949e","ytick.color":"#8b949e","text.color":"#e6edf3",
    "grid.color":"#21262d","grid.alpha":0.7,"lines.linewidth":2,
    "font.family":"DejaVu Sans","axes.titlesize":13,"axes.labelsize":11,
    "legend.framealpha":0.3,"legend.edgecolor":"#30363d",
})
ACCENT = ["#58a6ff","#3fb950","#f78166","#d2a8ff","#ffa657","#79c0ff","#ff7b72","#56d364"]
CMAP   = LinearSegmentedColormap.from_list("rag",["#161b22","#58a6ff","#3fb950"])
print("✅ Libraries loaded.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  ⚙️  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY  = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")
DEFAULT_MODEL      = "claude-sonnet-4-20250514"
DEFAULT_MAX_TOKENS = 1024
API_URL            = "https://api.anthropic.com/v1/messages"

# Cost config (USD per 1M tokens) — update for your model
COST_CONFIG = {
    "claude-sonnet-4-20250514": {"input": 3.00,  "output": 15.00},
    "claude-haiku-4-5-20251001":{"input": 0.80,  "output": 4.00},
    "claude-opus-4-6":          {"input": 15.00, "output": 75.00},
    "embedding-model":          {"input": 0.10,  "output": 0.00},  # per 1M tokens
}
ACTIVE_MODEL = "claude-sonnet-4-20250514"

# ── Shared call tracker ───────────────────────────────────────────────────────
@dataclass
class CostTracker:
    input_tokens : int = 0
    output_tokens: int = 0
    calls        : int = 0
    history      : List[Dict] = field(default_factory=list)

    def log(self, label, in_tok, out_tok, latency_ms=0):
        self.input_tokens  += in_tok
        self.output_tokens += out_tok
        self.calls         += 1
        model = ACTIVE_MODEL
        in_cost  = in_tok  / 1_000_000 * COST_CONFIG[model]["input"]
        out_cost = out_tok / 1_000_000 * COST_CONFIG[model]["output"]
        self.history.append({"label":label,"in":in_tok,"out":out_tok,
                              "cost_usd":in_cost+out_cost,"ms":latency_ms})

    @property
    def total_cost_usd(self):
        return sum(h["cost_usd"] for h in self.history)

    def summary(self):
        print(f"\n💰 Cost Tracker | calls={self.calls} | "
              f"in={self.input_tokens:,} | out={self.output_tokens:,} tok | "
              f"total=${self.total_cost_usd:.4f}")

TRACKER = CostTracker()

def call_llm(prompt, system="", max_tokens=DEFAULT_MAX_TOKENS,
             temperature=0.0, label="call", verbose=False):
    headers = {"x-api-key":ANTHROPIC_API_KEY,
               "anthropic-version":"2023-06-01","content-type":"application/json"}
    body = {"model":ACTIVE_MODEL,"max_tokens":max_tokens,"temperature":temperature,
            "messages":[{"role":"user","content":prompt}]}
    if system: body["system"] = system
    t0 = time.time()
    try:
        r = requests.post(API_URL, headers=headers, json=body, timeout=60)
        r.raise_for_status(); data = r.json()
    except Exception as e:
        return {"text":f"[ERROR:{e}]","in_tok":0,"out_tok":0,"ms":0}
    ms = int((time.time()-t0)*1000)
    text = data["content"][0]["text"]
    in_tok, out_tok = data["usage"]["input_tokens"], data["usage"]["output_tokens"]
    TRACKER.log(label, in_tok, out_tok, ms)
    if verbose: print(f"[{label}] {ms}ms | {in_tok}→{out_tok} tok\n{text[:300]}")
    return {"text":text,"in_tok":in_tok,"out_tok":out_tok,"ms":ms}

print("✅ API config ready. Key:", "SET ✅" if ANTHROPIC_API_KEY != "YOUR_API_KEY_HERE" else "NOT SET ⚠️")


In [ ]:
# ─── Synthetic Knowledge Base & RAG Pipeline ─────────────────────────────────
# A realistic KB covering multiple topics so we can test retrieval precision/recall

KNOWLEDGE_BASE = [
    # ── Vector Databases ─────────────────────────────────────────────────────
    {"id":"vdb_001","topic":"vector_db","text":"Pinecone is a managed vector database that supports HNSW indexing and real-time upserts. It provides sub-millisecond query latency at scale and supports metadata filtering combined with ANN search. Pinecone charges per vector stored and per query unit."},
    {"id":"vdb_002","topic":"vector_db","text":"Weaviate is an open-source vector database with a GraphQL API. It supports hybrid search combining dense and sparse vectors (BM25 + vector). Weaviate can be self-hosted or used as a managed cloud service."},
    {"id":"vdb_003","topic":"vector_db","text":"HNSW (Hierarchical Navigable Small World) is the dominant indexing algorithm for approximate nearest-neighbor search. It uses a multi-layer graph structure to achieve logarithmic search complexity. The ef_construction parameter controls build quality while ef_search controls query-time recall."},
    {"id":"vdb_004","topic":"vector_db","text":"Qdrant is an open-source vector database written in Rust, offering high performance and a rich filtering API. It supports on-disk indexing for large collections and provides a Python client SDK."},
    {"id":"vdb_005","topic":"vector_db","text":"ChromaDB is an embedded vector database designed for AI applications. It runs in-process with Python and is ideal for prototyping. ChromaDB supports persistent storage and can be upgraded to a client-server architecture."},

    # ── Embeddings ────────────────────────────────────────────────────────────
    {"id":"emb_001","topic":"embeddings","text":"text-embedding-3-large from OpenAI produces 3072-dimensional vectors and achieves state-of-the-art performance on the MTEB benchmark. It supports dimension reduction via the dimensions parameter without retraining."},
    {"id":"emb_002","topic":"embeddings","text":"Sentence transformers are BERT-based models fine-tuned using contrastive learning for semantic similarity tasks. The all-MiniLM-L6-v2 model is a popular lightweight option producing 384-dimensional embeddings with fast inference."},
    {"id":"emb_003","topic":"embeddings","text":"Embedding model context windows vary significantly. OpenAI's text-embedding-3-large supports 8191 tokens, while older models like Ada-002 also support 8191 tokens. Exceeding the context window causes silent truncation in most implementations."},
    {"id":"emb_004","topic":"embeddings","text":"Matryoshka Representation Learning (MRL) trains embeddings that remain useful at reduced dimensions. This allows trading retrieval quality for storage efficiency by truncating embedding vectors."},

    # ── RAG Architecture ──────────────────────────────────────────────────────
    {"id":"rag_001","topic":"rag","text":"Retrieval-Augmented Generation (RAG) combines a retrieval system with a generative language model. At inference time, relevant documents are fetched from an external knowledge base and injected into the LLM context as grounding evidence."},
    {"id":"rag_002","topic":"rag","text":"Naive RAG follows a simple retrieve-then-generate pattern: embed query, retrieve top-K chunks, concatenate as context, generate answer. Advanced RAG adds query rewriting, re-ranking, and iterative retrieval to improve quality."},
    {"id":"rag_003","topic":"rag","text":"HyDE (Hypothetical Document Embeddings) improves retrieval by having the LLM generate a hypothetical answer, embedding that answer, and using it as the retrieval query instead of the original question. This bridges the lexical gap between questions and documents."},
    {"id":"rag_004","topic":"rag","text":"Re-ranking is a two-stage retrieval pattern: first retrieve a large set (top-50) using fast ANN search, then re-rank with a cross-encoder model that scores query-document pairs jointly. Cross-encoders are slower but more accurate than bi-encoders."},
    {"id":"rag_005","topic":"rag","text":"Context window management in RAG requires careful chunk sizing. Stuffing too many chunks causes the lost-in-the-middle problem where LLMs ignore information in the middle of long contexts. Optimal chunk count is typically 3-7 for most models."},

    # ── Chunking ──────────────────────────────────────────────────────────────
    {"id":"chk_001","topic":"chunking","text":"Fixed-size chunking splits text every N characters regardless of semantic boundaries. It is fast and deterministic but often cuts sentences mid-way. Typical sizes are 256-1024 tokens with 10-15% overlap to preserve boundary context."},
    {"id":"chk_002","topic":"chunking","text":"Semantic chunking uses embedding similarity between adjacent sentences to detect topic shifts and insert chunk boundaries. It produces semantically coherent chunks but requires an embedding model during indexing."},
    {"id":"chk_003","topic":"chunking","text":"The chunk size vs retrieval quality trade-off: smaller chunks have higher retrieval precision but may miss context. Larger chunks have more context but lower precision. 512 tokens with overlap is a widely-used starting point."},

    # ── Evaluation ────────────────────────────────────────────────────────────
    {"id":"eval_001","topic":"evaluation","text":"RAGAS is an open-source framework for evaluating RAG pipelines using LLM-based metrics. Its core metrics are Faithfulness (factual consistency), Answer Relevancy, Context Precision, and Context Recall, each scored 0-1."},
    {"id":"eval_002","topic":"evaluation","text":"Faithfulness measures whether all claims in the generated answer are supported by the retrieved context. It is computed by extracting atomic claims from the answer and verifying each against the context using an LLM."},
    {"id":"eval_003","topic":"evaluation","text":"NDCG (Normalized Discounted Cumulative Gain) is an information retrieval metric that accounts for both relevance and ranking position. It gives higher credit for relevant documents appearing at rank 1 vs rank 10."},
    {"id":"eval_004","topic":"evaluation","text":"Mean Reciprocal Rank (MRR) measures retrieval quality by averaging the reciprocal of the rank at which the first relevant document appears. MRR=1.0 if the relevant doc is always rank 1, MRR=0.5 if always rank 2."},
    {"id":"eval_005","topic":"evaluation","text":"ROUGE (Recall-Oriented Understudy for Gisting Evaluation) measures n-gram overlap between generated and reference text. ROUGE-1 counts unigrams, ROUGE-2 bigrams, and ROUGE-L the longest common subsequence."},
]

# ── Ground-truth QA pairs ─────────────────────────────────────────────────────
GROUND_TRUTH_QA = [
    {
        "query"           : "What is HNSW and what parameters control its performance?",
        "relevant_doc_ids": ["vdb_003"],
        "answer_gt"       : "HNSW is a graph-based approximate nearest-neighbor indexing algorithm. ef_construction controls build quality and ef_search controls query-time recall.",
        "topic"           : "vector_db",
    },
    {
        "query"           : "How does HyDE improve RAG retrieval?",
        "relevant_doc_ids": ["rag_003"],
        "answer_gt"       : "HyDE improves retrieval by generating a hypothetical answer first, then using its embedding as the retrieval query instead of the original question.",
        "topic"           : "rag",
    },
    {
        "query"           : "What is the difference between bi-encoders and cross-encoders in re-ranking?",
        "relevant_doc_ids": ["rag_004"],
        "answer_gt"       : "Bi-encoders encode query and document independently for fast ANN search. Cross-encoders score query-document pairs jointly and are slower but more accurate.",
        "topic"           : "rag",
    },
    {
        "query"           : "What chunk size should I start with for RAG?",
        "relevant_doc_ids": ["chk_003","chk_001"],
        "answer_gt"       : "A common starting point is 512 tokens with 10-15% overlap. The trade-off is that smaller chunks have higher precision but less context.",
        "topic"           : "chunking",
    },
    {
        "query"           : "What are the core RAGAS evaluation metrics?",
        "relevant_doc_ids": ["eval_001","eval_002"],
        "answer_gt"       : "RAGAS core metrics are Faithfulness, Answer Relevancy, Context Precision, and Context Recall, each scored 0-1.",
        "topic"           : "evaluation",
    },
    {
        "query"           : "What is Matryoshka Representation Learning?",
        "relevant_doc_ids": ["emb_004"],
        "answer_gt"       : "MRL trains embeddings that remain useful at reduced dimensions, allowing dimension truncation to trade quality for storage efficiency.",
        "topic"           : "embeddings",
    },
    {
        "query"           : "How does ChromaDB differ from Pinecone?",
        "relevant_doc_ids": ["vdb_005","vdb_001"],
        "answer_gt"       : "ChromaDB is an embedded in-process database for prototyping while Pinecone is a managed cloud service with production-grade SLAs.",
        "topic"           : "vector_db",
    },
    {
        "query"           : "What is the lost-in-the-middle problem in RAG?",
        "relevant_doc_ids": ["rag_005"],
        "answer_gt"       : "LLMs tend to ignore information placed in the middle of long contexts. In RAG this means optimal context should use 3-7 chunks to avoid this degradation.",
        "topic"           : "rag",
    },
    {
        "query"           : "What is ROUGE and what variants exist?",
        "relevant_doc_ids": ["eval_005"],
        "answer_gt"       : "ROUGE measures n-gram overlap between generated and reference text. ROUGE-1 counts unigrams, ROUGE-2 bigrams, and ROUGE-L measures longest common subsequence.",
        "topic"           : "evaluation",
    },
    {
        "query"           : "What embedding model context window should I be aware of?",
        "relevant_doc_ids": ["emb_003"],
        "answer_gt"       : "Most embedding models support around 8191 tokens. Exceeding this causes silent truncation, which can degrade retrieval quality without obvious errors.",
        "topic"           : "embeddings",
    },
]

print(f"📚 Knowledge Base: {len(KNOWLEDGE_BASE)} documents across {len(set(d['topic'] for d in KNOWLEDGE_BASE))} topics")
print(f"❓ Ground-Truth QA Pairs: {len(GROUND_TRUTH_QA)}")
topic_counts = collections.Counter(d['topic'] for d in KNOWLEDGE_BASE)
for topic, count in topic_counts.items():
    print(f"   {topic:<15} {count} docs")


In [ ]:
# ─── Synthetic Retriever (TF-IDF + Cosine Similarity) ────────────────────────
# Simulates a vector retriever. Replace embed() with real embeddings in production.

class SyntheticRetriever:
    """
    TF-IDF based retriever that simulates semantic search.
    Swap embed_fn for real sentence-transformers in production.
    """
    def __init__(self, documents: List[Dict]):
        self.docs   = documents
        self.corpus = [d["text"] for d in documents]
        self.ids    = [d["id"]   for d in documents]
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
        self.matrix = self.vectorizer.fit_transform(self.corpus)
        print(f"  ✅ Retriever indexed {len(documents)} docs | vocab={self.vectorizer.vocabulary_.__len__()} terms")

    def retrieve(self, query: str, k: int = 5, with_scores: bool = True) -> List[Dict]:
        """Retrieve top-K documents for a query with similarity scores."""
        q_vec = self.vectorizer.transform([query])
        sims  = cosine_similarity(q_vec, self.matrix)[0]
        top_k = np.argsort(sims)[::-1][:k]
        results = []
        for rank, idx in enumerate(top_k):
            doc = self.docs[idx].copy()
            doc["score"] = float(sims[idx])
            doc["rank"]  = rank + 1
            results.append(doc)
        return results

    def retrieve_ids(self, query: str, k: int = 5) -> List[str]:
        return [d["id"] for d in self.retrieve(query, k)]


class SyntheticGenerator:
    """Wraps the LLM to generate answers from retrieved context."""

    def generate(self, query: str, context_docs: List[Dict],
                 max_tokens: int = 300) -> Dict:
        context_text = "\n\n".join([
            f"[Doc {i+1} | {d['id']}]\n{d['text']}"
            for i, d in enumerate(context_docs)
        ])
        prompt = f"""Answer the question using ONLY the provided context documents.
If the answer is not in the context, say "I don't have enough information."

Context:
{context_text}

Question: {query}

Answer:"""
        t0 = time.time()
        r  = call_llm(prompt, label="generate", max_tokens=max_tokens, temperature=0.0)
        return {
            "answer"    : r["text"],
            "in_tok"    : r["in_tok"],
            "out_tok"   : r["out_tok"],
            "latency_ms": r["ms"],
            "context"   : context_text,
        }


# ── Instantiate pipeline ──────────────────────────────────────────────────────
print("🔧 Building RAG Pipeline...")
RETRIEVER  = SyntheticRetriever(KNOWLEDGE_BASE)
GENERATOR  = SyntheticGenerator()

# Quick smoke test
test_r = RETRIEVER.retrieve("What is HNSW?", k=3)
print(f"\n  Smoke test — 'What is HNSW?' → top result: [{test_r[0]['id']}] score={test_r[0]['score']:.3f}")
print(f"  Text preview: {test_r[0]['text'][:80]}...")
print("\n✅ RAG pipeline ready.")


## 2. Retrieval Quality Metrics <a id='2-retrieval'></a>

Retrieval is the foundation — if the wrong documents are retrieved, even a perfect LLM can't save the answer. These metrics evaluate whether the **right documents** are being fetched at the **right rank positions**.

| Metric | What it measures | Range |
|--------|-----------------|-------|
| **Hit Rate@K** | Did ANY relevant doc appear in top-K? | 0–1 |
| **Precision@K** | Of top-K retrieved, what fraction is relevant? | 0–1 |
| **Recall@K** | Of all relevant docs, what fraction was in top-K? | 0–1 |
| **MRR** | How high is the FIRST relevant doc ranked? | 0–1 |
| **NDCG@K** | Recall + position quality combined | 0–1 |
| **MAP@K** | Mean Average Precision across all queries | 0–1 |


In [ ]:
# ─── Retrieval Metric Implementations ────────────────────────────────────────

def hit_rate_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    """
    Hit Rate@K: 1 if any relevant document appears in the top-K results, else 0.
    Also called Recall@1 in binary relevance settings.
    
    Intuition: 'Did the retriever surface AT LEAST ONE useful document?'
    """
    return 1.0 if any(rid in relevant_ids for rid in retrieved_ids[:k]) else 0.0


def precision_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    """
    Precision@K: fraction of top-K retrieved docs that are relevant.
    
    Intuition: 'How much of what I retrieved is actually useful?'
    Penalizes retrieving many irrelevant docs alongside relevant ones.
    """
    retrieved_k = retrieved_ids[:k]
    hits = sum(1 for rid in retrieved_k if rid in relevant_ids)
    return hits / k if k > 0 else 0.0


def recall_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    """
    Recall@K: fraction of all relevant docs that appear in top-K.
    
    Intuition: 'Did I find all the important documents?'
    Penalizes missing relevant documents even if retrieved ones are good.
    """
    if not relevant_ids: return 0.0
    hits = sum(1 for rid in retrieved_ids[:k] if rid in relevant_ids)
    return hits / len(relevant_ids)


def mean_reciprocal_rank(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
    """
    MRR: reciprocal of the rank of the first relevant document.
    
    MRR = 1/rank_of_first_hit
    If first hit is rank 1 → 1.0, rank 2 → 0.5, rank 3 → 0.33
    
    Intuition: 'How quickly does the retriever surface a useful document?'
    """
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    """
    NDCG@K: Normalized Discounted Cumulative Gain.
    
    Rewards relevant documents appearing higher in the ranking.
    A relevant doc at rank 1 contributes more than at rank 5.
    
    DCG  = sum(relevance_i / log2(rank_i + 1))
    IDCG = DCG of ideal (perfect) ranking
    NDCG = DCG / IDCG
    """
    dcg = 0.0
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant_ids:
            dcg += 1.0 / math.log2(rank + 1)

    # Ideal DCG: all relevant docs at top positions
    ideal_hits = min(len(relevant_ids), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))

    return dcg / idcg if idcg > 0 else 0.0


def average_precision_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    """
    AP@K: Average Precision — average of Precision@i for each rank i where doc_i is relevant.
    
    Mean AP@K across queries = MAP@K.
    Penalizes both missing relevant docs and ranking them low.
    """
    hits, precision_sum = 0, 0.0
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant_ids:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / min(len(relevant_ids), k) if relevant_ids else 0.0


def compute_retrieval_metrics(qa_pairs: List[Dict], retriever, k_values: List[int] = [1,3,5,10]) -> pd.DataFrame:
    """
    Run all retrieval metrics across all QA pairs and K values.
    Returns a DataFrame with one row per (query, K) combination.
    """
    rows = []
    for qa in qa_pairs:
        query        = qa["query"]
        relevant_ids = qa["relevant_doc_ids"]
        retrieved    = retriever.retrieve_ids(query, k=max(k_values))

        for k in k_values:
            rows.append({
                "query"     : query[:60],
                "topic"     : qa.get("topic",""),
                "k"         : k,
                "n_relevant": len(relevant_ids),
                "hit_rate"  : hit_rate_at_k(retrieved, relevant_ids, k),
                "precision" : precision_at_k(retrieved, relevant_ids, k),
                "recall"    : recall_at_k(retrieved, relevant_ids, k),
                "mrr"       : mean_reciprocal_rank(retrieved, relevant_ids),
                "ndcg"      : ndcg_at_k(retrieved, relevant_ids, k),
                "ap"        : average_precision_at_k(retrieved, relevant_ids, k),
            })
    return pd.DataFrame(rows)


# ── Run ───────────────────────────────────────────────────────────────────────
print("🔍 Computing retrieval metrics across all QA pairs...")
retrieval_df = compute_retrieval_metrics(GROUND_TRUTH_QA, RETRIEVER, k_values=[1,3,5,10])

# Summary by K
summary_by_k = retrieval_df.groupby("k")[["hit_rate","precision","recall","mrr","ndcg","ap"]].mean().round(3)
print("\n📊 Retrieval Metrics Summary (mean across all queries):")
print(summary_by_k.to_string())

map_score = retrieval_df[retrieval_df["k"]==5]["ap"].mean()
print(f"\n  MAP@5 = {map_score:.3f}")


In [ ]:
# ─── Retrieval Metrics Visualization ─────────────────────────────────────────

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Retrieval Quality Metrics — Full Analysis", fontsize=15, fontweight="bold")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

metric_cols = ["hit_rate","precision","recall","mrr","ndcg","ap"]
metric_labels = ["Hit Rate","Precision","Recall","MRR","NDCG","AP"]
k_vals = [1,3,5,10]

# Panel 1: Metrics vs K
ax1 = fig.add_subplot(gs[0,:2])
for i, (col, label) in enumerate(zip(metric_cols, metric_labels)):
    means = [retrieval_df[retrieval_df["k"]==k][col].mean() for k in k_vals]
    ax1.plot(k_vals, means, marker="o", color=ACCENT[i], label=label, linewidth=2)
ax1.set_xlabel("K (number of retrieved docs)")
ax1.set_ylabel("Score")
ax1.set_title("All Retrieval Metrics vs K")
ax1.legend(fontsize=8, ncol=3); ax1.grid(True, alpha=0.4)
ax1.set_xticks(k_vals); ax1.set_ylim(0,1.05)

# Panel 2: Per-query heatmap at K=5
ax2 = fig.add_subplot(gs[0,2])
k5  = retrieval_df[retrieval_df["k"]==5][["query","hit_rate","precision","recall","ndcg"]].set_index("query")
short_idx = [q[:30]+"…" for q in k5.index]
k5_plot   = k5.copy(); k5_plot.index = short_idx
im = ax2.imshow(k5_plot.values, cmap=CMAP, aspect="auto", vmin=0, vmax=1)
ax2.set_xticks(range(len(k5_plot.columns))); ax2.set_xticklabels(k5_plot.columns, fontsize=7, rotation=25)
ax2.set_yticks(range(len(k5_plot))); ax2.set_yticklabels(short_idx, fontsize=6)
ax2.set_title("Per-Query Scores @ K=5")
plt.colorbar(im, ax=ax2, shrink=0.7)

# Panel 3: Precision-Recall curve across K
ax3 = fig.add_subplot(gs[1,0])
p_vals = [retrieval_df[retrieval_df["k"]==k]["precision"].mean() for k in k_vals]
r_vals = [retrieval_df[retrieval_df["k"]==k]["recall"].mean()    for k in k_vals]
ax3.plot(r_vals, p_vals, "o-", color=ACCENT[0], linewidth=2)
for k, p, r in zip(k_vals, p_vals, r_vals):
    ax3.annotate(f"K={k}", (r, p), textcoords="offset points", xytext=(5,3), fontsize=8, color="white")
ax3.set_xlabel("Recall"); ax3.set_ylabel("Precision")
ax3.set_title("Precision-Recall Trade-off"); ax3.grid(True, alpha=0.4)
ax3.set_xlim(0,1.05); ax3.set_ylim(0,1.05)

# Panel 4: Topic-level performance at K=5
ax4 = fig.add_subplot(gs[1,1])
topic_perf = retrieval_df[retrieval_df["k"]==5].groupby("topic")["ndcg"].mean().sort_values()
colors_t = [ACCENT[i%len(ACCENT)] for i in range(len(topic_perf))]
bars = ax4.barh(topic_perf.index, topic_perf.values, color=colors_t, alpha=0.85, height=0.6)
ax4.set_xlabel("NDCG@5"); ax4.set_title("NDCG@5 by Topic")
ax4.set_xlim(0,1.1); ax4.grid(True, axis="x", alpha=0.4)
for bar, val in zip(bars, topic_perf.values):
    ax4.text(val+0.01, bar.get_y()+bar.get_height()/2, f"{val:.2f}", va="center", fontsize=9, color="white")

# Panel 5: MRR distribution
ax5 = fig.add_subplot(gs[1,2])
mrr_vals = retrieval_df[retrieval_df["k"]==10]["mrr"].values
ax5.hist(mrr_vals, bins=8, color=ACCENT[3], alpha=0.85, edgecolor="#0d1117")
ax5.axvline(mrr_vals.mean(), color="white", linestyle="--", linewidth=2, label=f"Mean={mrr_vals.mean():.2f}")
ax5.set_xlabel("MRR Score"); ax5.set_ylabel("Count")
ax5.set_title("MRR Distribution Across Queries"); ax5.legend(); ax5.grid(True, alpha=0.4)

plt.savefig("/tmp/retrieval_metrics.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"\n✅ Retrieval evaluation complete.")
print(f"   Best metric: Hit Rate@5 = {retrieval_df[retrieval_df['k']==5]['hit_rate'].mean():.3f}")
print(f"   Weakest:     Precision@5 = {retrieval_df[retrieval_df['k']==5]['precision'].mean():.3f}")
print(f"   → Implication: retriever finds relevant docs but also fetches noise")


## 3. Generation Quality Metrics <a id='3-generation'></a>

These metrics evaluate the **LLM's output** — assuming retrieval already happened. They measure whether the answer is accurate, complete, and well-formed.

| Metric | What it measures | Needs Reference? | Needs LLM? |
|--------|-----------------|-----------------|------------|
| **ROUGE-1/2/L** | N-gram overlap with reference | ✅ Yes | ❌ No |
| **Exact Match** | Perfect string match | ✅ Yes | ❌ No |
| **F1 Token Match** | Token-level F1 vs reference | ✅ Yes | ❌ No |
| **Semantic Similarity** | Cosine similarity of embeddings | ✅ Yes | ❌ No |
| **Answer Length** | Verbosity / conciseness signal | ❌ No | ❌ No |
| **Faithfulness** | Claims grounded in context | ❌ No | ✅ Yes |
| **Answer Relevancy** | Answer addresses the question | ❌ No | ✅ Yes |


In [ ]:
# ─── ROUGE Score Implementation ──────────────────────────────────────────────

def tokenize(text: str) -> List[str]:
    """Simple whitespace + punctuation tokenizer."""
    return re.findall(r'\b\w+\b', text.lower())

def ngrams(tokens: List[str], n: int) -> collections.Counter:
    """Return Counter of n-grams from token list."""
    return collections.Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def rouge_n(hypothesis: str, reference: str, n: int) -> Dict[str, float]:
    """
    Compute ROUGE-N: n-gram overlap between hypothesis and reference.
    
    Returns precision, recall, and F1.
    
    ROUGE-1: unigram overlap  — measures word coverage
    ROUGE-2: bigram overlap   — measures phrase coverage (stricter)
    """
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)
    hyp_ngrams = ngrams(hyp_tokens, n)
    ref_ngrams = ngrams(ref_tokens, n)

    # Count overlapping n-grams (clipped by reference count)
    overlap = sum((hyp_ngrams & ref_ngrams).values())
    precision = overlap / sum(hyp_ngrams.values()) if hyp_ngrams else 0.0
    recall    = overlap / sum(ref_ngrams.values())  if ref_ngrams else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


def rouge_l(hypothesis: str, reference: str) -> Dict[str, float]:
    """
    ROUGE-L: Longest Common Subsequence (LCS) based F1.
    Captures sentence-level structure better than ROUGE-N.
    """
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)

    m, n = len(hyp_tokens), len(ref_tokens)
    # LCS via dynamic programming
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if hyp_tokens[i-1] == ref_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs_len   = dp[m][n]
    precision = lcs_len / m if m > 0 else 0.0
    recall    = lcs_len / n if n > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


def token_f1(hypothesis: str, reference: str) -> float:
    """
    Token-level F1: harmonic mean of token precision and recall.
    Used in SQuAD evaluation — handles paraphrasing better than exact match.
    """
    hyp_tokens = collections.Counter(tokenize(hypothesis))
    ref_tokens = collections.Counter(tokenize(reference))
    common     = sum((hyp_tokens & ref_tokens).values())
    if common == 0: return 0.0
    precision  = common / sum(hyp_tokens.values())
    recall     = common / sum(ref_tokens.values())
    return 2 * precision * recall / (precision + recall)


def semantic_similarity(text1: str, text2: str, vectorizer=None) -> float:
    """
    Compute TF-IDF cosine similarity as a proxy for semantic similarity.
    In production: replace with actual embedding model (sentence-transformers).
    """
    vect = vectorizer or TfidfVectorizer(ngram_range=(1,2))
    try:
        matrix = vect.fit_transform([text1, text2])
        return float(cosine_similarity(matrix[0], matrix[1])[0][0])
    except:
        return 0.0


def exact_match(hypothesis: str, reference: str) -> float:
    """Normalized exact match — strips punctuation and case."""
    def normalize(s):
        s = s.lower().strip()
        s = re.sub(r'[^\w\s]', '', s)
        s = re.sub(r'\s+', ' ', s)
        return s
    return 1.0 if normalize(hypothesis) == normalize(reference) else 0.0


# ── Run generation metrics on generated answers ───────────────────────────────
print("📝 Generating answers + computing generation metrics...")
print("   (This makes API calls — one per QA pair)")
print()

generation_results = []
for qa in GROUND_TRUTH_QA:
    # Retrieve context
    retrieved_docs = RETRIEVER.retrieve(qa["query"], k=3)
    # Generate answer
    gen = GENERATOR.generate(qa["query"], retrieved_docs, max_tokens=200)
    hyp = gen["answer"]
    ref = qa["answer_gt"]

    r1 = rouge_n(hyp, ref, 1)
    r2 = rouge_n(hyp, ref, 2)
    rl = rouge_l(hyp, ref)

    generation_results.append({
        "query"      : qa["query"][:55],
        "topic"      : qa["topic"],
        "hypothesis" : hyp,
        "reference"  : ref,
        "rouge1_f1"  : round(r1["f1"],3),
        "rouge2_f1"  : round(r2["f1"],3),
        "rougeL_f1"  : round(rl["f1"],3),
        "token_f1"   : round(token_f1(hyp, ref),3),
        "exact_match": round(exact_match(hyp, ref),3),
        "sem_sim"    : round(semantic_similarity(hyp, ref),3),
        "answer_len" : len(hyp.split()),
        "in_tok"     : gen["in_tok"],
        "out_tok"    : gen["out_tok"],
        "latency_ms" : gen["latency_ms"],
    })
    print(f"  ✅ {qa['query'][:50]}")

gen_df = pd.DataFrame(generation_results)
print("\n📊 Generation Quality Summary:")
print(gen_df[["rouge1_f1","rouge2_f1","rougeL_f1","token_f1","sem_sim"]].describe().round(3).to_string())


In [ ]:
# ─── Generation Metrics Visualization ────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Generation Quality Metrics — Full Analysis", fontsize=15, fontweight="bold")

# Panel 1: Metric distributions (violin)
ax = axes[0,0]
metric_cols = ["rouge1_f1","rouge2_f1","rougeL_f1","token_f1","sem_sim"]
data = [gen_df[c].values for c in metric_cols]
vp = ax.violinplot(data, positions=range(len(metric_cols)), showmeans=True, showmedians=True)
for i, pc in enumerate(vp["bodies"]):
    pc.set_facecolor(ACCENT[i]); pc.set_alpha(0.6)
vp["cmeans"].set_color("white"); vp["cmedians"].set_color("#ffa657")
ax.set_xticks(range(len(metric_cols)))
ax.set_xticklabels(["R1","R2","RL","TokF1","SemSim"], fontsize=9)
ax.set_ylabel("Score"); ax.set_title("Metric Distributions"); ax.grid(True, axis="y", alpha=0.4)
ax.set_ylim(0,1.1)

# Panel 2: Per-query heatmap
ax = axes[0,1]
hm_data = gen_df[["rouge1_f1","rouge2_f1","rougeL_f1","token_f1","sem_sim"]].values
im = ax.imshow(hm_data.T, cmap=CMAP, aspect="auto", vmin=0, vmax=1)
ax.set_yticks(range(5)); ax.set_yticklabels(["R1","R2","RL","TokF1","SemSim"], fontsize=8)
ax.set_xticks(range(len(gen_df))); ax.set_xticklabels(range(1, len(gen_df)+1))
ax.set_xlabel("Query #"); ax.set_title("Per-Query Score Heatmap")
plt.colorbar(im, ax=ax, shrink=0.8)

# Panel 3: ROUGE-1 vs Semantic Similarity scatter
ax = axes[0,2]
ax.scatter(gen_df["rouge1_f1"], gen_df["sem_sim"], 
           c=[ACCENT[i%len(ACCENT)] for i in range(len(gen_df))], s=80, zorder=5, alpha=0.9)
for i, row in gen_df.iterrows():
    ax.annotate(f"Q{i+1}", (row["rouge1_f1"], row["sem_sim"]),
                textcoords="offset points", xytext=(4,3), fontsize=7, color="white")
corr, _ = pearsonr(gen_df["rouge1_f1"], gen_df["sem_sim"])
ax.set_xlabel("ROUGE-1 F1"); ax.set_ylabel("Semantic Similarity")
ax.set_title(f"ROUGE-1 vs Semantic Sim (r={corr:.2f})"); ax.grid(True, alpha=0.4)

# Panel 4: By topic
ax = axes[1,0]
topic_gen = gen_df.groupby("topic")[["rouge1_f1","rougeL_f1","sem_sim"]].mean()
x = np.arange(len(topic_gen))
w = 0.25
for i, (col, label) in enumerate(zip(["rouge1_f1","rougeL_f1","sem_sim"],["R1","RL","SemSim"])):
    ax.bar(x + (i-1)*w, topic_gen[col], w, alpha=0.85, color=ACCENT[i], label=label)
ax.set_xticks(x); ax.set_xticklabels(topic_gen.index, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("Score"); ax.set_title("Generation Quality by Topic")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.4); ax.set_ylim(0,1.1)

# Panel 5: Answer length distribution
ax = axes[1,1]
ax.hist(gen_df["answer_len"], bins=8, color=ACCENT[4], alpha=0.85, edgecolor="#0d1117")
ax.axvline(gen_df["answer_len"].mean(), color="white", linestyle="--",
           label=f"Mean={gen_df['answer_len'].mean():.0f}w")
ax.set_xlabel("Answer Length (words)"); ax.set_ylabel("Count")
ax.set_title("Answer Length Distribution"); ax.legend(); ax.grid(True, alpha=0.4)

# Panel 6: Metric correlation heatmap
ax = axes[1,2]
corr_matrix = gen_df[metric_cols].corr()
im2 = ax.imshow(corr_matrix.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(5)); ax.set_xticklabels(["R1","R2","RL","TokF1","SemSim"], fontsize=8)
ax.set_yticks(range(5)); ax.set_yticklabels(["R1","R2","RL","TokF1","SemSim"], fontsize=8)
ax.set_title("Metric Correlation Matrix")
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{corr_matrix.values[i,j]:.2f}", ha="center", va="center",
                fontsize=8, color="black", fontweight="bold")
plt.colorbar(im2, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig("/tmp/generation_metrics.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("\n💡 Key insight: ROUGE and Semantic Similarity don't always agree.")
print("   High corr = paraphrase of reference. Low corr = different wording but same meaning.")
print("   → Always use BOTH lexical (ROUGE) and semantic metrics together.")


## 4. End-to-End RAG Metrics <a id='4-e2e'></a>

These metrics evaluate the **full pipeline** — retrieval + generation together. They capture how well the system performs as a whole, not just individual components.

| Metric | Formula | What it catches |
|--------|---------|-----------------|
| **Context Recall** | Are relevant docs in retrieved context? | Missing retrieval |
| **Context Precision** | Are retrieved docs actually used in answer? | Noise in retrieval |
| **Answer Correctness** | Weighted combo of F1 + semantic sim | Overall answer quality |
| **Answer Completeness** | Does answer cover all key facts from reference? | Incomplete answers |
| **Groundedness** | Are claims in answer traceable to context? | Hallucination |


In [ ]:
# ─── E2E RAG Metrics ──────────────────────────────────────────────────────────

def context_recall(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
    """
    Context Recall: fraction of relevant documents that appear in the retrieved context.
    
    If all ground-truth relevant docs were retrieved → 1.0
    If none were retrieved → 0.0
    
    A low context recall means the retriever MISSED important documents.
    The generator cannot compensate for missing context.
    """
    if not relevant_ids: return 1.0
    hits = sum(1 for rid in relevant_ids if rid in retrieved_ids)
    return hits / len(relevant_ids)


def context_precision(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
    """
    Context Precision: fraction of retrieved documents that are relevant.
    
    High precision = clean context, low noise
    Low precision  = retriever adds irrelevant docs that may confuse the LLM
    
    Note: This is identical to Precision@K in retrieval metrics but framed
    from the context quality perspective.
    """
    if not retrieved_ids: return 0.0
    hits = sum(1 for rid in retrieved_ids if rid in relevant_ids)
    return hits / len(retrieved_ids)


def answer_correctness(hypothesis: str, reference: str, 
                       semantic_weight: float = 0.4) -> float:
    """
    Answer Correctness: weighted combination of Token F1 and Semantic Similarity.
    
    correctness = (1-w) * token_f1 + w * semantic_similarity
    
    Uses both lexical and semantic comparison to handle paraphrasing.
    Default: 60% lexical (Token F1) + 40% semantic (cosine sim).
    """
    tf1 = token_f1(hypothesis, reference)
    sem = semantic_similarity(hypothesis, reference)
    return (1 - semantic_weight) * tf1 + semantic_weight * sem


def answer_completeness(hypothesis: str, reference: str,
                        n_key_phrases: int = 5) -> float:
    """
    Answer Completeness: fraction of key phrases from the reference that
    appear in the hypothesis.
    
    Extracts top-N content words from reference as 'key facts' and 
    checks which appear in the hypothesis.
    """
    # Extract content words (noun-like tokens > 4 chars) as key facts
    stop_words = {"this","that","with","from","they","have","what","when",
                  "which","will","been","their","there","were","about","into"}
    ref_words   = [w for w in tokenize(reference) if len(w) > 4 and w not in stop_words]
    hyp_words   = set(tokenize(hypothesis))

    if not ref_words: return 0.0
    # Use most frequent words from reference as key facts
    key_facts   = [w for w, _ in collections.Counter(ref_words).most_common(n_key_phrases)]
    hits        = sum(1 for kf in key_facts if kf in hyp_words)
    return hits / len(key_facts)


def groundedness_score(hypothesis: str, context: str) -> float:
    """
    Groundedness (heuristic): fraction of hypothesis tokens that appear
    in the context (lexical overlap proxy).
    
    High groundedness = answer stays close to context language
    Low groundedness  = answer may include hallucinated content
    
    Note: For production use Section 6's LLM-based hallucination detector.
    """
    hyp_tokens  = set(tokenize(hypothesis))
    ctx_tokens  = set(tokenize(context))
    if not hyp_tokens: return 0.0
    overlap     = hyp_tokens & ctx_tokens
    return len(overlap) / len(hyp_tokens)


# ── Full E2E evaluation run ───────────────────────────────────────────────────
print("🔄 Running End-to-End RAG evaluation...")
e2e_results = []

for i, (qa, gen_row) in enumerate(zip(GROUND_TRUTH_QA, generation_results)):
    retrieved_ids = RETRIEVER.retrieve_ids(qa["query"], k=5)
    retrieved_docs = RETRIEVER.retrieve(qa["query"], k=5)
    context_text   = " ".join(d["text"] for d in retrieved_docs)

    cr  = context_recall(retrieved_ids, qa["relevant_doc_ids"])
    cp  = context_precision(retrieved_ids, qa["relevant_doc_ids"])
    ac  = answer_correctness(gen_row["hypothesis"], qa["answer_gt"])
    acomp = answer_completeness(gen_row["hypothesis"], qa["answer_gt"])
    grnd  = groundedness_score(gen_row["hypothesis"], context_text)

    e2e_results.append({
        "query"               : qa["query"][:55],
        "topic"               : qa["topic"],
        "context_recall"      : round(cr,3),
        "context_precision"   : round(cp,3),
        "answer_correctness"  : round(ac,3),
        "answer_completeness" : round(acomp,3),
        "groundedness"        : round(grnd,3),
        "rag_score"           : round(np.mean([cr, cp, ac, acomp, grnd]),3),
    })
    print(f"  Q{i+1} | CR={cr:.2f} CP={cp:.2f} AC={ac:.2f} COMP={acomp:.2f} GND={grnd:.2f}")

e2e_df = pd.DataFrame(e2e_results)
print("\n📊 E2E RAG Metrics Summary:")
print(e2e_df[["context_recall","context_precision","answer_correctness",
              "answer_completeness","groundedness","rag_score"]].describe().round(3).to_string())


In [ ]:
# ─── E2E Metrics Visualization ───────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("End-to-End RAG Metrics Dashboard", fontsize=15, fontweight="bold")

e2e_metrics = ["context_recall","context_precision","answer_correctness",
               "answer_completeness","groundedness"]
e2e_labels  = ["Context\nRecall","Context\nPrecision","Answer\nCorrectness",
               "Answer\nCompleteness","Groundedness"]

# Panel 1: Overall RAG score per query (bar)
ax = axes[0,0]
queries_short = [f"Q{i+1}" for i in range(len(e2e_df))]
colors_score  = [ACCENT[2] if s >= 0.7 else ACCENT[4] if s >= 0.5 else ACCENT[2+5]
                 for s in e2e_df["rag_score"]]
bars = ax.bar(queries_short, e2e_df["rag_score"], color=colors_score, alpha=0.85, width=0.7)
ax.axhline(e2e_df["rag_score"].mean(), color="white", linestyle="--",
           label=f"Mean={e2e_df['rag_score'].mean():.2f}")
ax.set_ylabel("RAG Score (mean of 5 metrics)"); ax.set_title("Overall RAG Score per Query")
ax.set_ylim(0,1.1); ax.legend(); ax.grid(True, axis="y", alpha=0.4)
for bar, val in zip(bars, e2e_df["rag_score"]):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f"{val:.2f}",
            ha="center", fontsize=8, color="white")

# Panel 2: Radar chart per query
ax = axes[0,1]
angles = np.linspace(0, 2*np.pi, len(e2e_metrics), endpoint=False).tolist() + [0]
mean_vals = [e2e_df[m].mean() for m in e2e_metrics] + [e2e_df[e2e_metrics[0]].mean()]
ax2_polar = fig.add_subplot(2,2,2,polar=True)
axes[0,1].remove()
ax2_polar.set_facecolor("#161b22")
ax2_polar.plot(angles, mean_vals, "o-", color=ACCENT[0], linewidth=2, label="Mean")
ax2_polar.fill(angles, mean_vals, alpha=0.2, color=ACCENT[0])
# Best and worst query
best_q  = e2e_df.loc[e2e_df["rag_score"].idxmax()]
worst_q = e2e_df.loc[e2e_df["rag_score"].idxmin()]
for q_row, label, color in [(best_q,"Best",ACCENT[1]),(worst_q,"Worst",ACCENT[2])]:
    vals = [q_row[m] for m in e2e_metrics] + [q_row[e2e_metrics[0]]]
    ax2_polar.plot(angles, vals, "o--", color=color, linewidth=1.5, alpha=0.8, label=label)
ax2_polar.set_xticks(angles[:-1]); ax2_polar.set_xticklabels(e2e_labels, fontsize=8, color="#e6edf3")
ax2_polar.set_ylim(0,1); ax2_polar.set_title("E2E Metric Radar", color="#e6edf3", pad=20)
ax2_polar.legend(loc="upper right", bbox_to_anchor=(1.3,1.1), fontsize=8)
ax2_polar.grid(color="#30363d")

# Panel 3: Context Recall vs Precision
ax = axes[1,0]
sc = ax.scatter(e2e_df["context_recall"], e2e_df["context_precision"],
                c=e2e_df["answer_correctness"], cmap=CMAP, s=100, zorder=5, vmin=0, vmax=1)
for i, row in e2e_df.iterrows():
    ax.annotate(f"Q{i+1}", (row["context_recall"], row["context_precision"]),
                textcoords="offset points", xytext=(4,3), fontsize=8, color="white")
plt.colorbar(sc, ax=ax, label="Answer Correctness")
ax.set_xlabel("Context Recall"); ax.set_ylabel("Context Precision")
ax.set_title("Context Recall vs Precision\n(color = Answer Correctness)")
ax.set_xlim(-0.05,1.1); ax.set_ylim(-0.05,1.1); ax.grid(True, alpha=0.4)
ax.axhline(0.5, color="white", linestyle=":", alpha=0.4)
ax.axvline(0.5, color="white", linestyle=":", alpha=0.4)
ax.text(0.6,0.6,"✅ Ideal Region",fontsize=9,color=ACCENT[1],alpha=0.8)

# Panel 4: Topic-level E2E heatmap
ax = axes[1,1]
topic_e2e = e2e_df.groupby("topic")[e2e_metrics].mean()
im = ax.imshow(topic_e2e.values, cmap=CMAP, aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(e2e_metrics))); ax.set_xticklabels(e2e_labels, fontsize=7)
ax.set_yticks(range(len(topic_e2e))); ax.set_yticklabels(topic_e2e.index, fontsize=9)
ax.set_title("E2E Metrics by Topic")
plt.colorbar(im, ax=ax, shrink=0.8)
for i in range(topic_e2e.shape[0]):
    for j in range(topic_e2e.shape[1]):
        ax.text(j, i, f"{topic_e2e.values[i,j]:.2f}", ha="center", va="center",
                fontsize=8, color="white", fontweight="bold")

plt.tight_layout()
plt.savefig("/tmp/e2e_metrics.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()


## 5. LLM-as-Judge Evaluation <a id='5-llm-judge'></a>

LLM-as-judge uses a powerful model to score RAG outputs on human-aligned rubrics — capturing nuance that n-gram metrics miss. This is now standard in production RAG evaluation.

**Frameworks covered:**
- **Single-answer scoring:** Rubric-based 1-10 score with critique
- **Pairwise comparison:** Which of A vs B is better?
- **G-Eval:** Structured rubric with explicit scoring steps (best correlation with human judgments)
- **RAGAS-style faithfulness:** Claim extraction + per-claim verification


In [ ]:
# ─── LLM-as-Judge: Single Answer Scoring ─────────────────────────────────────

def llm_judge_single(
    query   : str,
    answer  : str,
    context : str,
    criteria: Dict[str, str] = None,
) -> Dict:
    """
    Score a RAG answer on multiple criteria using an LLM judge.
    
    Default criteria: faithfulness, relevance, completeness, clarity.
    Returns per-criterion scores (0-10) + reasoning + overall score.
    """
    default_criteria = {
        "faithfulness"  : "Are all claims in the answer supported by the provided context? Penalize any claim not in context.",
        "relevance"     : "Does the answer directly address the question asked? Penalize off-topic content.",
        "completeness"  : "Does the answer cover all key points needed to fully answer the question?",
        "clarity"       : "Is the answer clear, well-structured, and easy to understand?",
    }
    criteria = criteria or default_criteria

    criteria_str = "\n".join([f"- **{k}** (0-10): {v}" for k,v in criteria.items()])
    keys_list    = list(criteria.keys())

    prompt = f"""You are an expert RAG system evaluator. Score the following answer.

QUESTION: {query}

RETRIEVED CONTEXT:
{context[:1500]}

GENERATED ANSWER:
{answer}

Score the answer on each criterion (0-10 integer):
{criteria_str}

Respond ONLY with this JSON (no markdown):
{{
{chr(10).join([f'  "{k}": <0-10>,' for k in keys_list])}
  "overall": <0-10>,
  "reasoning": "<2-3 sentence critique>",
  "main_weakness": "<single biggest issue>"
}}"""

    r = call_llm(prompt, label="llm-judge-single", max_tokens=400, temperature=0.0)
    try:
        # Strip any accidental markdown
        clean = re.sub(r"```json|```", "", r["text"]).strip()
        scores = json.loads(clean)
    except:
        scores = {k: 5 for k in keys_list}
        scores.update({"overall":5,"reasoning":"Parse error","main_weakness":"N/A"})
    
    scores["in_tok"]  = r["in_tok"]
    scores["out_tok"] = r["out_tok"]
    return scores


# ── Run LLM judge on all QA pairs ────────────────────────────────────────────
print("⚖️  Running LLM-as-Judge evaluation...")
print("   (Makes 1 API call per query)")
judge_results = []

for i, (qa, gen_row) in enumerate(zip(GROUND_TRUTH_QA, generation_results)):
    retrieved_docs = RETRIEVER.retrieve(qa["query"], k=5)
    context_text   = "\n".join([f"[{d['id']}] {d['text']}" for d in retrieved_docs])
    scores = llm_judge_single(qa["query"], gen_row["hypothesis"], context_text)
    
    scores["query"]  = qa["query"][:55]
    scores["topic"]  = qa["topic"]
    scores["answer"] = gen_row["hypothesis"][:100]
    judge_results.append(scores)
    print(f"  Q{i+1} | overall={scores['overall']}/10 | {scores.get('main_weakness','')[:50]}")

judge_df = pd.DataFrame(judge_results)
print("\n📊 LLM Judge Summary (mean scores out of 10):")
score_cols = ["faithfulness","relevance","completeness","clarity","overall"]
print(judge_df[score_cols].mean().round(2).to_string())


In [ ]:
# ─── G-Eval: Structured Step-by-Step Scoring ─────────────────────────────────

def g_eval(
    query   : str,
    answer  : str,
    context : str,
    dimension: str = "faithfulness",
) -> Dict:
    """
    G-Eval: Generate explicit evaluation steps, then score.
    
    G-Eval (Liu et al. 2023) has highest correlation with human judgments
    because it forces the model to reason through a rubric before scoring.
    
    Two-stage process:
    1. Generate evaluation steps specific to the dimension
    2. Follow those steps to produce a score
    """
    # Stage 1: Generate evaluation steps
    step_gen_prompt = f"""You are designing an evaluation rubric.

Task: Evaluate a RAG system answer for '{dimension}'.
Dimension definition:
- faithfulness: Every claim in the answer must be traceable to the context
- relevance: The answer must directly address what was asked
- completeness: The answer must cover all necessary information

Generate 4 specific, concrete evaluation steps for '{dimension}'.
Return ONLY a numbered list, no other text."""

    r_steps = call_llm(step_gen_prompt, label="g-eval-steps", max_tokens=200, temperature=0.0)
    steps   = r_steps["text"]

    # Stage 2: Score following the generated steps
    score_prompt = f"""Follow these evaluation steps to score the answer:

{steps}

Now evaluate:
QUESTION: {query}
CONTEXT: {context[:1000]}
ANSWER: {answer}

After following the steps above, respond ONLY with JSON:
{{"score": <1-10>, "step_scores": [<score per step>], "verdict": "<pass/fail/partial>"}}"""

    r_score = call_llm(score_prompt, label="g-eval-score", max_tokens=150, temperature=0.0)
    try:
        clean   = re.sub(r"```json|```","", r_score["text"]).strip()
        result  = json.loads(clean)
    except:
        result  = {"score":5,"step_scores":[5,5,5,5],"verdict":"partial"}

    result["steps"]    = steps
    result["in_tok"]   = r_steps["in_tok"] + r_score["in_tok"]
    result["out_tok"]  = r_steps["out_tok"] + r_score["out_tok"]
    return result


# ── Demo G-Eval on first 3 QA pairs ──────────────────────────────────────────
print("🎯 G-Eval: Structured Step-by-Step Scoring")
print("="*65)

g_eval_results = []
for i, (qa, gen_row) in enumerate(list(zip(GROUND_TRUTH_QA, generation_results))[:3]):
    retrieved_docs = RETRIEVER.retrieve(qa["query"], k=3)
    context_text   = " ".join(d["text"] for d in retrieved_docs)
    
    result = g_eval(qa["query"], gen_row["hypothesis"], context_text, "faithfulness")
    g_eval_results.append(result)
    
    print(f"\nQ{i+1}: {qa['query'][:60]}")
    print(f"  G-Eval Score: {result['score']}/10 | Verdict: {result['verdict']}")
    print(f"  Steps used:\n{result['steps'][:200]}...")


In [ ]:
# ─── Pairwise Comparison Evaluation ─────────────────────────────────────────

def pairwise_judge(
    query    : str,
    answer_a : str,
    answer_b : str,
    context  : str,
    label_a  : str = "System A",
    label_b  : str = "System B",
) -> Dict:
    """
    Pairwise LLM judge: which answer is better, A or B?
    
    Pairwise comparison has higher inter-annotator agreement than 
    absolute scoring because it reduces position bias.
    
    Returns: winner, confidence, reasoning, per-criterion preferences.
    """
    prompt = f"""Compare two RAG system answers to the same question.

QUESTION: {query}

CONTEXT:
{context[:1000]}

ANSWER A ({label_a}):
{answer_a}

ANSWER B ({label_b}):
{answer_b}

Evaluate on: faithfulness, relevance, completeness, clarity.
Pick a winner or declare a tie.

Respond ONLY with JSON:
{{
  "winner": "A" | "B" | "tie",
  "confidence": "high" | "medium" | "low",
  "preferred_on": {{
    "faithfulness": "A" | "B" | "tie",
    "relevance"   : "A" | "B" | "tie",
    "completeness": "A" | "B" | "tie",
    "clarity"     : "A" | "B" | "tie"
  }},
  "reasoning": "<2 sentences explaining the winner>"
}}"""

    r = call_llm(prompt, label="pairwise-judge", max_tokens=300, temperature=0.0)
    try:
        clean  = re.sub(r"```json|```","",r["text"]).strip()
        result = json.loads(clean)
    except:
        result = {"winner":"tie","confidence":"low","preferred_on":{},"reasoning":"Parse error"}
    result["in_tok"]  = r["in_tok"]
    result["out_tok"] = r["out_tok"]
    return result


# ── Demo: Compare RAG-generated answers vs shorter truncated versions ─────────
print("⚔️  Pairwise Comparison: Full RAG Answer vs Truncated Baseline")
print("="*65)

pairwise_results = []
for i, (qa, gen_row) in enumerate(list(zip(GROUND_TRUTH_QA, generation_results))[:3]):
    retrieved_docs  = RETRIEVER.retrieve(qa["query"], k=3)
    context_text    = " ".join(d["text"] for d in retrieved_docs)
    
    answer_a        = gen_row["hypothesis"]                        # Full RAG answer
    answer_b        = " ".join(gen_row["hypothesis"].split()[:20]) + "..."  # Truncated baseline

    result = pairwise_judge(qa["query"], answer_a, answer_b, context_text,
                             "Full RAG", "Truncated Baseline")
    pairwise_results.append(result)
    print(f"\nQ{i+1}: {qa['query'][:55]}")
    print(f"  Winner: {result['winner']} | Confidence: {result['confidence']}")
    print(f"  Reasoning: {result['reasoning'][:120]}")

# Summarize wins
winners = [r["winner"] for r in pairwise_results]
print(f"\n📊 Win summary: A={winners.count('A')} | B={winners.count('B')} | Tie={winners.count('tie')}")


## 6. Hallucination Detection <a id='6-hallucination'></a>

Hallucination — the model stating facts not in the retrieved context — is the most critical failure mode in production RAG. This section implements a full hallucination detection pipeline.

**Pipeline:**
1. **Claim extraction** — decompose the answer into atomic factual claims
2. **Per-claim verification** — check each claim against context (NLI-style)
3. **Hallucination rate** — fraction of claims not grounded in context
4. **Severity classification** — minor drift vs major fabrication


In [ ]:
# ─── Hallucination Detection Pipeline ────────────────────────────────────────

def extract_claims(answer: str) -> List[str]:
    """
    Stage 1: Extract atomic factual claims from an answer using an LLM.
    Each claim should be a single verifiable statement.
    """
    prompt = f"""Extract all atomic factual claims from this text.
Each claim must be a single, verifiable statement — not a question or opinion.
If the text is vague or contains no verifiable facts, return an empty list.

Text: {answer}

Return ONLY a JSON list of claim strings:
["claim 1", "claim 2", ...]"""

    r = call_llm(prompt, label="claim-extract", max_tokens=300, temperature=0.0)
    try:
        clean  = re.sub(r"```json|```","",r["text"]).strip()
        claims = json.loads(clean)
        if isinstance(claims, list):
            return [str(c) for c in claims if c]
    except:
        # Fallback: split on periods
        sentences = [s.strip() for s in answer.split(".") if len(s.strip()) > 15]
        return sentences[:5]
    return []


def verify_claim(claim: str, context: str) -> Dict:
    """
    Stage 2: Verify whether a single claim is supported by the context.
    
    Returns:
        verdict   : "supported" | "contradicted" | "not_in_context"
        confidence: "high" | "medium" | "low"
        evidence  : excerpt from context supporting/refuting the claim
    """
    prompt = f"""Determine if this claim is supported by the provided context.

CONTEXT:
{context[:1500]}

CLAIM: {claim}

Classify as:
- "supported"       : The context explicitly supports this claim
- "contradicted"    : The context explicitly contradicts this claim
- "not_in_context"  : The context doesn't mention this — neither supports nor contradicts

Respond ONLY with JSON:
{{"verdict": "supported"|"contradicted"|"not_in_context",
  "confidence": "high"|"medium"|"low",
  "evidence": "<relevant excerpt from context, or 'none'>"}}"""

    r = call_llm(prompt, label="claim-verify", max_tokens=150, temperature=0.0)
    try:
        clean  = re.sub(r"```json|```","",r["text"]).strip()
        result = json.loads(clean)
    except:
        result = {"verdict":"not_in_context","confidence":"low","evidence":"parse error"}
    result["claim"] = claim
    return result


def hallucination_score(answer: str, context: str) -> Dict:
    """
    Full hallucination detection pipeline for one answer.
    
    Returns:
        hallucination_rate : fraction of claims not supported by context
        claims             : list of all extracted claims
        verifications      : per-claim verdict dicts
        severity           : "none" | "minor" | "moderate" | "severe"
    """
    claims = extract_claims(answer)
    if not claims:
        return {"hallucination_rate":0.0,"claims":[],"verifications":[],"severity":"none"}
    
    verifications = []
    for claim in claims:
        result = verify_claim(claim, context)
        verifications.append(result)
    
    supported   = sum(1 for v in verifications if v["verdict"]=="supported")
    contradicted= sum(1 for v in verifications if v["verdict"]=="contradicted")
    not_in_ctx  = sum(1 for v in verifications if v["verdict"]=="not_in_context")
    
    hallucination_rate = (contradicted + not_in_ctx) / len(claims)
    
    if hallucination_rate == 0:             severity = "none"
    elif hallucination_rate <= 0.2:         severity = "minor"
    elif hallucination_rate <= 0.5:         severity = "moderate"
    else:                                   severity = "severe"
    
    return {
        "hallucination_rate": round(hallucination_rate, 3),
        "n_claims"          : len(claims),
        "n_supported"       : supported,
        "n_contradicted"    : contradicted,
        "n_not_in_context"  : not_in_ctx,
        "claims"            : claims,
        "verifications"     : verifications,
        "severity"          : severity,
    }


# ── Run hallucination detection on first 5 QA pairs ───────────────────────────
print("🔬 Hallucination Detection Pipeline")
print("="*65)
print("   Extracting claims → verifying each claim against context\n")

halluc_results = []
for i, (qa, gen_row) in enumerate(list(zip(GROUND_TRUTH_QA, generation_results))[:5]):
    retrieved_docs = RETRIEVER.retrieve(qa["query"], k=3)
    context_text   = " ".join(d["text"] for d in retrieved_docs)
    
    result = hallucination_score(gen_row["hypothesis"], context_text)
    result["query"] = qa["query"][:55]
    result["topic"] = qa["topic"]
    halluc_results.append(result)
    
    severity_emoji = {"none":"✅","minor":"⚠️","moderate":"🟠","severe":"🔴"}
    print(f"Q{i+1}: {qa['query'][:55]}")
    print(f"  {severity_emoji.get(result['severity'],'?')} Severity={result['severity'].upper()} | "
          f"rate={result['hallucination_rate']:.0%} | "
          f"{result['n_claims']} claims: "
          f"{result['n_supported']} supported / {result['n_not_in_context']} not-in-context / "
          f"{result['n_contradicted']} contradicted")
    if result["verifications"]:
        for v in result["verifications"][:2]:
            status = "✅" if v["verdict"]=="supported" else "❌" if v["verdict"]=="contradicted" else "⚠️"
            print(f"    {status} '{v['claim'][:80]}' → {v['verdict']}")
    print()


In [ ]:
# ─── Hallucination Visualization ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Hallucination Detection Results", fontsize=14, fontweight="bold")

# Panel 1: Hallucination rate per query
ax = axes[0]
labels_h = [f"Q{i+1}" for i in range(len(halluc_results))]
rates    = [r["hallucination_rate"] for r in halluc_results]
colors_h = [ACCENT[1] if r==0 else ACCENT[4] if r<=0.2 else ACCENT[2] for r in rates]
bars = ax.bar(labels_h, rates, color=colors_h, alpha=0.85, width=0.6)
ax.set_ylabel("Hallucination Rate"); ax.set_title("Hallucination Rate per Query")
ax.set_ylim(0, 1.0); ax.axhline(0.2, color="white", linestyle="--", alpha=0.5, label="20% threshold")
ax.legend(); ax.grid(True, axis="y", alpha=0.4)
for bar, val in zip(bars, rates):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f"{val:.0%}",
            ha="center", fontsize=9, color="white")

# Panel 2: Verdict distribution (stacked bar)
ax = axes[1]
n_sup  = [r["n_supported"]     for r in halluc_results]
n_nic  = [r["n_not_in_context"]for r in halluc_results]
n_con  = [r["n_contradicted"]  for r in halluc_results]
x      = range(len(halluc_results))
ax.bar(x, n_sup, color=ACCENT[1], label="Supported",       alpha=0.85)
ax.bar(x, n_nic, bottom=n_sup, color=ACCENT[4],  label="Not in Context", alpha=0.85)
ax.bar(x, n_con, bottom=[s+n for s,n in zip(n_sup,n_nic)], color=ACCENT[2], label="Contradicted", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels_h)
ax.set_ylabel("Number of Claims"); ax.set_title("Claim Verdict Distribution")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.4)

# Panel 3: Severity distribution pie
ax = axes[2]
severities = [r["severity"] for r in halluc_results]
sev_counts = collections.Counter(severities)
sev_colors = {"none":ACCENT[1],"minor":ACCENT[4],"moderate":"#ffa657","severe":ACCENT[2]}
wedge_colors = [sev_colors.get(s, ACCENT[0]) for s in sev_counts.keys()]
wedges, texts, autotexts = ax.pie(
    sev_counts.values(), labels=sev_counts.keys(),
    autopct="%1.0f%%", colors=wedge_colors,
    textprops={"color":"white","fontsize":10})
for at in autotexts: at.set_fontsize(9)
ax.set_title("Hallucination Severity Distribution")

plt.tight_layout()
plt.savefig("/tmp/hallucination.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("\n💡 Production threshold recommendation:")
print("   • Hallucination rate > 20% → trigger human review")
print("   • 'contradicted' verdict → immediate escalation (LLM is actively wrong)")
print("   • 'not_in_context' verdict → may be benign (model using parametric knowledge)")


## 7. Latency & Performance Profiling <a id='7-latency'></a>

Production RAG has strict latency SLAs. Every millisecond matters. This section profiles each pipeline stage, computes P50/P95/P99 percentiles, and identifies bottlenecks.

**Typical RAG latency budget breakdown:**

| Stage | Typical Time | % of Total |
|-------|-------------|------------|
| Query embedding | 10–50ms | 2–5% |
| Vector retrieval (ANN) | 5–20ms | 1–3% |
| Re-ranking (optional) | 100–500ms | 10–20% |
| Context assembly | 1–5ms | <1% |
| LLM generation | 500–3000ms | 70–90% |
| Response parsing | 1–5ms | <1% |


In [ ]:
# ─── RAG Pipeline Profiler ───────────────────────────────────────────────────

@dataclass
class StageTimer:
    """Context manager for timing individual pipeline stages."""
    name    : str
    timings : List[float] = field(default_factory=list)

    def time(self, fn, *args, **kwargs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        self.timings.append(elapsed_ms)
        return result, elapsed_ms

    def stats(self) -> Dict:
        if not self.timings: return {}
        arr = np.array(self.timings)
        return {
            "stage"  : self.name,
            "n"      : len(arr),
            "mean_ms": round(float(np.mean(arr)),  2),
            "p50_ms" : round(float(np.percentile(arr, 50)), 2),
            "p95_ms" : round(float(np.percentile(arr, 95)), 2),
            "p99_ms" : round(float(np.percentile(arr, 99)), 2),
            "min_ms" : round(float(np.min(arr)),   2),
            "max_ms" : round(float(np.max(arr)),   2),
            "std_ms" : round(float(np.std(arr)),   2),
        }


class ProfiledRAGPipeline:
    """Wraps the RAG pipeline with per-stage timing instrumentation."""

    def __init__(self, retriever, generator):
        self.retriever = retriever
        self.generator = generator
        # One timer per stage
        self.timers = {
            "embedding"   : StageTimer("embedding"),
            "retrieval"   : StageTimer("retrieval"),
            "reranking"   : StageTimer("reranking"),
            "context_assm": StageTimer("context_assembly"),
            "generation"  : StageTimer("generation"),
            "total"       : StageTimer("total"),
        }

    def run(self, query: str, k: int = 5, rerank: bool = True) -> Dict:
        """Run a full profiled RAG query."""
        trace = {}
        t_total = time.perf_counter()

        # Stage 1: Embedding (simulated — TF-IDF transform)
        _, t_emb = self.timers["embedding"].time(
            lambda q: self.retriever.vectorizer.transform([q]), query)
        trace["embedding_ms"] = t_emb

        # Stage 2: Retrieval (ANN search)
        _, t_ret = self.timers["retrieval"].time(
            self.retriever.retrieve, query, k)
        docs = self.retriever.retrieve(query, k)
        trace["retrieval_ms"] = t_ret

        # Stage 3: Re-ranking (simulated: sort by score + small perturbation)
        if rerank:
            def mock_rerank(docs):
                time.sleep(0.05 + np.random.exponential(0.03))  # simulate 50-80ms cross-encoder
                return sorted(docs, key=lambda d: d["score"] + np.random.normal(0,0.01), reverse=True)
            docs, t_rer = self.timers["reranking"].time(mock_rerank, docs)
            trace["reranking_ms"] = t_rer
        else:
            trace["reranking_ms"] = 0.0

        # Stage 4: Context assembly
        def assemble(docs):
            return "\n\n".join([f"[{d['id']}] {d['text']}" for d in docs[:3]])
        context, t_ctx = self.timers["context_assm"].time(assemble, docs)
        trace["context_assembly_ms"] = t_ctx

        # Stage 5: LLM Generation
        gen_result, t_gen = self.timers["generation"].time(
            self.generator.generate, query, docs[:3], 200)
        trace["generation_ms"]  = t_gen
        trace["answer"]         = gen_result["answer"]
        trace["in_tok"]         = gen_result["in_tok"]
        trace["out_tok"]        = gen_result["out_tok"]

        # Total
        trace["total_ms"] = (time.perf_counter() - t_total) * 1000
        self.timers["total"].timings.append(trace["total_ms"])
        return trace

    def stage_stats(self) -> pd.DataFrame:
        return pd.DataFrame([t.stats() for t in self.timers.values() if t.timings])


# ── Profile 5 queries ─────────────────────────────────────────────────────────
print("⏱  Profiling RAG Pipeline (5 queries, with re-ranking)")
print("="*65)

profiler = ProfiledRAGPipeline(RETRIEVER, GENERATOR)
profile_traces = []

test_queries = [qa["query"] for qa in GROUND_TRUTH_QA[:5]]
for i, query in enumerate(test_queries):
    trace = profiler.run(query, k=5, rerank=True)
    profile_traces.append(trace)
    print(f"  Q{i+1} | total={trace['total_ms']:.0f}ms | "
          f"emb={trace['embedding_ms']:.1f}ms | "
          f"ret={trace['retrieval_ms']:.1f}ms | "
          f"rer={trace['reranking_ms']:.0f}ms | "
          f"gen={trace['generation_ms']:.0f}ms")

stats_df = profiler.stage_stats()
print("\n📊 Stage Statistics:")
print(stats_df[["stage","mean_ms","p50_ms","p95_ms","p99_ms","max_ms"]].to_string(index=False))


In [ ]:
# ─── Latency Visualization ───────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("RAG Latency & Performance Profiling", fontsize=15, fontweight="bold")

stages = ["embedding","retrieval","reranking","context_assembly","generation"]
stage_labels = ["Embedding","Retrieval","Re-ranking","Context\nAssembly","Generation"]
stage_keys   = ["embedding_ms","retrieval_ms","reranking_ms","context_assembly_ms","generation_ms"]

# Panel 1: Stacked bar — latency breakdown per query
ax = axes[0,0]
bottoms = np.zeros(len(profile_traces))
for i, (key, label) in enumerate(zip(stage_keys, stage_labels)):
    vals = [t.get(key, 0) for t in profile_traces]
    ax.bar(range(len(profile_traces)), vals, bottom=bottoms,
           color=ACCENT[i], label=label, alpha=0.85)
    bottoms += np.array(vals)
ax.set_xticks(range(len(profile_traces)))
ax.set_xticklabels([f"Q{i+1}" for i in range(len(profile_traces))])
ax.set_ylabel("Latency (ms)"); ax.set_title("Latency Breakdown per Query")
ax.legend(fontsize=8, loc="upper right"); ax.grid(True, axis="y", alpha=0.4)

# Panel 2: P50 / P95 / P99 per stage
ax = axes[0,1]
stats_filtered = stats_df[stats_df["stage"].isin(["embedding","retrieval","reranking",
                                                    "context_assembly","generation"])]
if not stats_filtered.empty:
    x = np.arange(len(stats_filtered))
    w = 0.25
    ax.bar(x-w, stats_filtered["p50_ms"], w, color=ACCENT[1], label="P50", alpha=0.85)
    ax.bar(x,   stats_filtered["p95_ms"], w, color=ACCENT[4], label="P95", alpha=0.85)
    ax.bar(x+w, stats_filtered["p99_ms"], w, color=ACCENT[2], label="P99", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(stats_filtered["stage"].str.replace("_"," ").str.title(), 
                       rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("ms"); ax.set_title("P50 / P95 / P99 per Stage")
    ax.legend(); ax.grid(True, axis="y", alpha=0.4)

# Panel 3: Generation latency vs token output (scatter)
ax = axes[1,0]
gen_ms  = [t["generation_ms"]  for t in profile_traces]
out_tok = [t["out_tok"]        for t in profile_traces]
ax.scatter(out_tok, gen_ms, c=ACCENT[0], s=100, zorder=5)
for i, (x, y) in enumerate(zip(out_tok, gen_ms)):
    ax.annotate(f"Q{i+1}", (x,y), textcoords="offset points", xytext=(4,3), fontsize=9, color="white")
ax.set_xlabel("Output Tokens"); ax.set_ylabel("Generation Latency (ms)")
ax.set_title("Token Count vs Generation Latency"); ax.grid(True, alpha=0.4)
# Fit line
if len(out_tok) > 1:
    m, b = np.polyfit(out_tok, gen_ms, 1)
    x_line = np.linspace(min(out_tok), max(out_tok), 50)
    ax.plot(x_line, m*x_line+b, "--", color=ACCENT[1], alpha=0.7, label=f"~{m:.1f}ms/tok")
    ax.legend()

# Panel 4: Time-to-first-token proxy (embedding + retrieval only)
ax = axes[1,1]
ttft = [t["embedding_ms"] + t["retrieval_ms"] for t in profile_traces]
total= [t["total_ms"] for t in profile_traces]
ax.bar(range(len(profile_traces)), total, color=ACCENT[0], alpha=0.5, label="Total", width=0.6)
ax.bar(range(len(profile_traces)), ttft,  color=ACCENT[1], alpha=0.85, label="Pre-LLM (TTFT proxy)", width=0.6)
ax.set_xticks(range(len(profile_traces)))
ax.set_xticklabels([f"Q{i+1}" for i in range(len(profile_traces))])
ax.set_ylabel("ms"); ax.set_title("Total Latency vs Pre-LLM Latency")
ax.legend(); ax.grid(True, axis="y", alpha=0.4)

# SLA reference line
sla_ms = 3000
ax.axhline(sla_ms, color=ACCENT[2], linestyle="--", alpha=0.7, label=f"SLA={sla_ms}ms")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/latency.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()

# Bottleneck report
print("\n🔍 Bottleneck Analysis:")
if not stats_df.empty:
    gen_pct = stats_df[stats_df["stage"]=="generation"]["mean_ms"].values
    total_pct = stats_df[stats_df["stage"]=="total"]["mean_ms"].values
    if len(gen_pct) and len(total_pct) and total_pct[0] > 0:
        print(f"   Generation accounts for {gen_pct[0]/total_pct[0]*100:.0f}% of total latency")
print("   → Optimization priority: streaming output, smaller model for simple queries,")
print("     caching frequent queries, async re-ranking")


## 8. Cost Analysis & Optimization <a id='8-cost'></a>

Cost is a first-class production metric. This section computes cost per query, full pipeline cost breakdowns, and models the ROI of different optimization strategies.

**Cost drivers in RAG:**
- **Embedding API calls** (indexing + query-time)
- **LLM input tokens** (grows with chunk size × K retrieved)
- **LLM output tokens** (grows with answer length)
- **Re-ranking model** (optional but adds cost)
- **Vector DB storage + queries** (infrastructure cost)


In [ ]:
# ─── Cost Calculator ─────────────────────────────────────────────────────────

def compute_token_cost(
    in_tokens : int,
    out_tokens: int,
    model     : str = "claude-sonnet-4-20250514",
) -> Dict:
    """
    Compute USD cost for a single LLM call.
    
    Returns:
        input_cost_usd  : Cost of input tokens
        output_cost_usd : Cost of output tokens
        total_cost_usd  : Total call cost
        cost_per_1k_in  : Per-1K-token input rate
        cost_per_1k_out : Per-1K-token output rate
    """
    if model not in COST_CONFIG:
        raise ValueError(f"Unknown model: {model}. Add to COST_CONFIG.")
    
    rate_in  = COST_CONFIG[model]["input"]  / 1_000_000
    rate_out = COST_CONFIG[model]["output"] / 1_000_000
    
    in_cost  = in_tokens  * rate_in
    out_cost = out_tokens * rate_out
    
    return {
        "model"          : model,
        "input_tokens"   : in_tokens,
        "output_tokens"  : out_tokens,
        "input_cost_usd" : round(in_cost,  6),
        "output_cost_usd": round(out_cost, 6),
        "total_cost_usd" : round(in_cost + out_cost, 6),
    }


def estimate_pipeline_cost(
    queries_per_day  : int,
    avg_chunk_tokens : int   = 512,
    k_retrieved      : int   = 5,
    avg_answer_tokens: int   = 200,
    embedding_model  : str   = "embedding-model",
    llm_model        : str   = "claude-sonnet-4-20250514",
    system_prompt_tok: int   = 200,
    query_tokens     : int   = 50,
    include_reranking: bool  = False,
    rerank_cost_per_q: float = 0.0001,
) -> Dict:
    """
    Estimate monthly cost for a RAG pipeline at scale.
    
    Args:
        queries_per_day   : Volume of queries
        avg_chunk_tokens  : Tokens per retrieved chunk
        k_retrieved       : Number of chunks injected into context
        avg_answer_tokens : Average LLM output length
        system_prompt_tok : Fixed system prompt overhead
        query_tokens      : Tokens in the user query
        include_reranking : Whether to add re-ranking cost
    
    Returns:
        Itemized cost dict per query and per month.
    """
    # Input token count = system prompt + query + K chunks + some formatting
    context_tokens = k_retrieved * avg_chunk_tokens
    total_in       = system_prompt_tok + query_tokens + context_tokens + 50  # 50 = overhead

    # Embedding cost per query (query embedding only; doc indexing is one-time)
    emb_cost = query_tokens / 1_000_000 * COST_CONFIG[embedding_model]["input"]

    # LLM cost per query
    llm  = compute_token_cost(total_in, avg_answer_tokens, llm_model)
    
    # Reranking cost (if applicable)
    rer_cost = rerank_cost_per_q if include_reranking else 0.0

    per_query = emb_cost + llm["total_cost_usd"] + rer_cost
    per_day   = per_query * queries_per_day
    per_month = per_day * 30
    per_year  = per_day * 365

    return {
        "config": {
            "queries_per_day"  : queries_per_day,
            "k_retrieved"      : k_retrieved,
            "avg_chunk_tokens" : avg_chunk_tokens,
            "avg_answer_tokens": avg_answer_tokens,
            "llm_model"        : llm_model,
        },
        "per_query": {
            "embedding_usd"   : round(emb_cost, 7),
            "llm_input_usd"   : round(llm["input_cost_usd"], 6),
            "llm_output_usd"  : round(llm["output_cost_usd"],6),
            "reranking_usd"   : round(rer_cost, 6),
            "total_usd"       : round(per_query, 6),
            "context_tokens"  : context_tokens,
            "total_in_tokens" : total_in,
        },
        "per_month_usd" : round(per_month, 2),
        "per_year_usd"  : round(per_year,  2),
        "per_day_usd"   : round(per_day,   2),
    }


# ── Model comparison at scale ─────────────────────────────────────────────────
print("💰 RAG Cost Analysis — Model × Volume Comparison")
print("="*65)

scenarios = [
    {"label":"Startup (500 q/day)",   "queries_per_day":500},
    {"label":"SMB (5K q/day)",        "queries_per_day":5_000},
    {"label":"Enterprise (50K q/day)","queries_per_day":50_000},
    {"label":"Scale (500K q/day)",    "queries_per_day":500_000},
]
models = ["claude-haiku-4-5-20251001","claude-sonnet-4-20250514","claude-opus-4-6"]

cost_rows = []
for sc in scenarios:
    for model in models:
        est = estimate_pipeline_cost(
            queries_per_day=sc["queries_per_day"],
            llm_model=model, k_retrieved=5,
            avg_chunk_tokens=512, avg_answer_tokens=200
        )
        cost_rows.append({
            "Scenario"      : sc["label"],
            "Model"         : model.replace("claude-","").replace("-20250514","").replace("-20251001",""),
            "Cost/Query ($)": est["per_query"]["total_usd"],
            "Cost/Day ($)"  : est["per_day_usd"],
            "Cost/Month ($)": est["per_month_usd"],
            "Cost/Year ($)" : est["per_year_usd"],
            "Context Tokens": est["per_query"]["context_tokens"],
        })

cost_df = pd.DataFrame(cost_rows)
print(cost_df[["Scenario","Model","Cost/Query ($)","Cost/Month ($)","Cost/Year ($)"]].to_string(index=False))


In [ ]:
# ─── Cost Optimization Strategies & ROI ──────────────────────────────────────

def cost_sensitivity_analysis(base_config: Dict, param: str, 
                               values: List, queries_per_day: int = 10_000) -> pd.DataFrame:
    """
    Analyze how changing a single parameter affects monthly cost.
    """
    rows = []
    for v in values:
        cfg = {**base_config, param: v}
        est = estimate_pipeline_cost(queries_per_day=queries_per_day, **cfg)
        rows.append({param: v, "monthly_cost": est["per_month_usd"],
                     "per_query_cost": est["per_query"]["total_usd"],
                     "context_tokens": est["per_query"]["context_tokens"]})
    return pd.DataFrame(rows)


base = {"k_retrieved":5,"avg_chunk_tokens":512,"avg_answer_tokens":200,
        "llm_model":"claude-sonnet-4-20250514"}

# Sensitivity analyses
k_sens    = cost_sensitivity_analysis(base, "k_retrieved",       [1,2,3,5,7,10])
chunk_sens= cost_sensitivity_analysis(base, "avg_chunk_tokens",  [128,256,512,1024,2048])
ans_sens  = cost_sensitivity_analysis(base, "avg_answer_tokens", [50,100,150,200,400,800])

# Model switching savings
haiku_est  = estimate_pipeline_cost(10_000, llm_model="claude-haiku-4-5-20251001")
sonnet_est = estimate_pipeline_cost(10_000, llm_model="claude-sonnet-4-20250514")
opus_est   = estimate_pipeline_cost(10_000, llm_model="claude-opus-4-6")

print("💡 Cost Optimization Impact (at 10K queries/day):")
print(f"   Haiku  vs Sonnet: save ${sonnet_est['per_month_usd']-haiku_est['per_month_usd']:,.0f}/month "
      f"({(1-haiku_est['per_month_usd']/sonnet_est['per_month_usd'])*100:.0f}% savings)")
print(f"   K=3    vs K=5  : save ${cost_sensitivity_analysis(base,'k_retrieved',[3,5],10_000)['monthly_cost'].diff().abs().iloc[-1]:,.0f}/month")
print(f"   Chunk 256 vs 512: save ${cost_sensitivity_analysis(base,'avg_chunk_tokens',[256,512],10_000)['monthly_cost'].diff().abs().iloc[-1]:,.0f}/month")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Cost Analysis & Optimization", fontsize=15, fontweight="bold")

# Panel 1: K sensitivity
ax = axes[0,0]
ax.plot(k_sens["k_retrieved"], k_sens["monthly_cost"], "o-", color=ACCENT[0], linewidth=2)
ax.fill_between(k_sens["k_retrieved"], k_sens["monthly_cost"], alpha=0.15, color=ACCENT[0])
ax.set_xlabel("K (chunks retrieved)"); ax.set_ylabel("Monthly Cost ($)")
ax.set_title("Cost vs K Retrieved (10K q/day)"); ax.grid(True, alpha=0.4)
for _, row in k_sens.iterrows():
    ax.annotate(f"K={int(row['k_retrieved'])}", (row["k_retrieved"], row["monthly_cost"]),
                textcoords="offset points", xytext=(3,5), fontsize=8, color="white")

# Panel 2: Chunk size sensitivity
ax = axes[0,1]
ax.plot(chunk_sens["avg_chunk_tokens"], chunk_sens["monthly_cost"], "s-", color=ACCENT[3], linewidth=2)
ax.fill_between(chunk_sens["avg_chunk_tokens"], chunk_sens["monthly_cost"], alpha=0.15, color=ACCENT[3])
ax.set_xlabel("Avg Chunk Size (tokens)"); ax.set_ylabel("Monthly Cost ($)")
ax.set_title("Cost vs Chunk Size (10K q/day)"); ax.grid(True, alpha=0.4)
ax.axvline(512, color="white", linestyle="--", alpha=0.5, label="Common default (512)")
ax.legend()

# Panel 3: Model cost at scale
ax = axes[1,0]
vol_range = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]
for model_name, color in zip(models, [ACCENT[1], ACCENT[0], ACCENT[2]]):
    costs = [estimate_pipeline_cost(v, llm_model=model_name)["per_month_usd"] for v in vol_range]
    label = model_name.replace("claude-","").replace("-20250514","").replace("-20251001","")
    ax.plot(vol_range, costs, "o-", color=color, label=label, linewidth=2)
ax.set_xlabel("Queries per Day"); ax.set_ylabel("Monthly Cost ($)")
ax.set_title("Monthly Cost by Model at Scale"); ax.legend(fontsize=8)
ax.set_xscale("log"); ax.set_yscale("log"); ax.grid(True, alpha=0.4, which="both")

# Panel 4: Cost breakdown waterfall for single query
ax = axes[1,1]
est_detail = estimate_pipeline_cost(10_000, llm_model="claude-sonnet-4-20250514",
                                     include_reranking=True)
components = ["Embedding","LLM Input","LLM Output","Re-ranking"]
values_usd  = [
    est_detail["per_query"]["embedding_usd"],
    est_detail["per_query"]["llm_input_usd"],
    est_detail["per_query"]["llm_output_usd"],
    est_detail["per_query"]["reranking_usd"],
]
colors_wf = [ACCENT[i] for i in range(len(components))]
bars = ax.bar(components, values_usd, color=colors_wf, alpha=0.85, width=0.6)
ax.set_ylabel("Cost per Query ($)"); ax.set_title("Cost per Query Breakdown (Sonnet)")
ax.grid(True, axis="y", alpha=0.4)
for bar, val in zip(bars, values_usd):
    ax.text(bar.get_x()+bar.get_width()/2, val+max(values_usd)*0.01,
            f"${val:.6f}", ha="center", fontsize=8, color="white")

plt.tight_layout()
plt.savefig("/tmp/cost_analysis.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"\n📋 Current session cost: ${TRACKER.total_cost_usd:.4f}")
TRACKER.summary()


## 9. Regression Testing Framework <a id='9-regression'></a>

Regression testing ensures that every change to your RAG pipeline (new model, new chunking, new prompt) is measured against a baseline before shipping to production.

**The framework:**
1. Establish a **baseline** with your current best configuration
2. Define **alert thresholds** for each metric
3. Run comparison on every pipeline change
4. Auto-flag regressions for human review


In [ ]:
# ─── Regression Test Framework ───────────────────────────────────────────────

@dataclass
class RAGBaseline:
    """Stores a snapshot of RAG performance metrics as a baseline."""
    name        : str
    timestamp   : str = field(default_factory=lambda: time.strftime("%Y-%m-%d %H:%M:%S"))
    metrics     : Dict[str, float] = field(default_factory=dict)
    config      : Dict[str, Any]   = field(default_factory=dict)

    def to_dict(self) -> Dict:
        return {"name":self.name,"timestamp":self.timestamp,
                "metrics":self.metrics,"config":self.config}


# Threshold config: (min_delta, direction)
# direction: "higher_better" → alert if new < baseline - delta
#            "lower_better"  → alert if new > baseline + delta
ALERT_THRESHOLDS = {
    "hit_rate@5"           : (-0.05, "higher_better"),
    "ndcg@5"               : (-0.05, "higher_better"),
    "mrr"                  : (-0.05, "higher_better"),
    "context_recall"       : (-0.05, "higher_better"),
    "context_precision"    : (-0.05, "higher_better"),
    "answer_correctness"   : (-0.05, "higher_better"),
    "faithfulness_judge"   : (-0.05, "higher_better"),
    "hallucination_rate"   : (0.05,  "lower_better"),
    "p95_latency_ms"       : (200,   "lower_better"),
    "cost_per_query_usd"   : (0.001, "lower_better"),
}


def run_eval_suite(qa_pairs, retriever, generator=None, k=5) -> Dict[str, float]:
    """Run a full evaluation suite and return a flat metrics dict."""
    metrics = {}
    
    # Retrieval metrics
    ret_df = compute_retrieval_metrics(qa_pairs, retriever, k_values=[k])
    metrics[f"hit_rate@{k}"]  = ret_df["hit_rate"].mean()
    metrics[f"ndcg@{k}"]      = ret_df["ndcg"].mean()
    metrics["mrr"]             = ret_df["mrr"].mean()
    metrics[f"precision@{k}"] = ret_df["precision"].mean()
    metrics[f"recall@{k}"]    = ret_df["recall"].mean()
    
    # E2E metrics
    e2e_rows = []
    for qa in qa_pairs:
        r_ids = retriever.retrieve_ids(qa["query"], k)
        r_docs = retriever.retrieve(qa["query"], k)
        ctx    = " ".join(d["text"] for d in r_docs)
        
        # Generate only if generator provided (skip to save cost)
        hyp = qa["answer_gt"][:100]  # default: use reference as hypothesis (no-API mode)
        if generator:
            gen = generator.generate(qa["query"], r_docs[:3], max_tokens=150)
            hyp = gen["answer"]
        
        e2e_rows.append({
            "context_recall"   : context_recall(r_ids, qa["relevant_doc_ids"]),
            "context_precision": context_precision(r_ids, qa["relevant_doc_ids"]),
            "answer_correctness": answer_correctness(hyp, qa["answer_gt"]),
            "groundedness"     : groundedness_score(hyp, ctx),
        })
    
    e2e_frame = pd.DataFrame(e2e_rows)
    for col in e2e_frame.columns:
        metrics[col] = e2e_frame[col].mean()
    
    # Cost
    cost_est = estimate_pipeline_cost(10_000, llm_model=ACTIVE_MODEL, k_retrieved=k)
    metrics["cost_per_query_usd"] = cost_est["per_query"]["total_usd"]
    
    return {k: round(v,4) for k,v in metrics.items()}


def compare_to_baseline(new_metrics: Dict, baseline: RAGBaseline) -> pd.DataFrame:
    """Compare new metrics against baseline and flag regressions."""
    rows = []
    for metric, new_val in new_metrics.items():
        if metric not in baseline.metrics: continue
        base_val = baseline.metrics[metric]
        delta    = new_val - base_val
        delta_pct= (delta / base_val * 100) if base_val != 0 else 0

        # Check alert threshold
        regression = False
        if metric in ALERT_THRESHOLDS:
            threshold, direction = ALERT_THRESHOLDS[metric]
            if direction == "higher_better" and delta < threshold:
                regression = True
            elif direction == "lower_better" and delta > abs(threshold):
                regression = True

        rows.append({
            "metric"    : metric,
            "baseline"  : round(base_val, 4),
            "new"       : round(new_val,  4),
            "delta"     : round(delta,    4),
            "delta_pct" : round(delta_pct,1),
            "regression": regression,
            "status"    : "🔴 REGRESSION" if regression else ("🟢 improved" if delta > 0 else "⚪ stable"),
        })
    return pd.DataFrame(rows)


# ── Build baseline + run comparison ──────────────────────────────────────────
print("📐 Building baseline (offline metrics — no extra API calls)...")
baseline_metrics = run_eval_suite(GROUND_TRUTH_QA, RETRIEVER, generator=None, k=5)

BASELINE = RAGBaseline(
    name    = "baseline_v1.0",
    metrics = baseline_metrics,
    config  = {"k":5,"model":ACTIVE_MODEL,"chunking":"fixed_512","reranking":False},
)

print(f"\n✅ Baseline established: '{BASELINE.name}' @ {BASELINE.timestamp}")
print("\n  Key baseline metrics:")
for k_m, v in list(baseline_metrics.items())[:8]:
    print(f"    {k_m:<30} {v:.4f}")

# Simulate a slightly degraded "new" system
print("\n\n🧪 Simulating pipeline change (K=3 instead of K=5)...")
new_metrics = run_eval_suite(GROUND_TRUTH_QA, RETRIEVER, generator=None, k=3)

comparison_df = compare_to_baseline(new_metrics, BASELINE)
print("\n📊 Regression Test Report:")
print(comparison_df[["metric","baseline","new","delta_pct","status"]].to_string(index=False))

n_regressions = comparison_df["regression"].sum()
print(f"\n  {'🔴 FAIL' if n_regressions > 0 else '✅ PASS'} — {n_regressions} regression(s) detected")


In [ ]:
# ─── Regression Visualization ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Regression Testing — Baseline vs New Pipeline", fontsize=14, fontweight="bold")

# Panel 1: Delta bar chart
ax = axes[0]
comp_plot = comparison_df.dropna(subset=["delta_pct"]).head(10)
colors_r  = [ACCENT[2] if r else ACCENT[1] for r in comp_plot["regression"]]
bars = ax.barh(comp_plot["metric"], comp_plot["delta_pct"], color=colors_r, alpha=0.85, height=0.6)
ax.axvline(0, color="white", linewidth=1, alpha=0.7)
ax.set_xlabel("Delta % (positive = improvement)")
ax.set_title("Metric Changes vs Baseline")
ax.grid(True, axis="x", alpha=0.4)
for bar, val in zip(bars, comp_plot["delta_pct"]):
    ax.text(val + (0.3 if val >= 0 else -0.3), bar.get_y()+bar.get_height()/2,
            f"{val:+.1f}%", va="center", ha="left" if val >= 0 else "right",
            fontsize=8, color="white")

# Add regression threshold line markers
ax.axvline(-5, color=ACCENT[2], linestyle="--", alpha=0.5, label="-5% regression threshold")
ax.legend(fontsize=8)

# Panel 2: Baseline vs New — grouped bar
ax = axes[1]
shared = comparison_df.dropna(subset=["baseline","new"]).head(8)
x  = np.arange(len(shared))
w  = 0.35
b1 = ax.bar(x-w/2, shared["baseline"], w, color=ACCENT[0], alpha=0.85, label="Baseline")
b2 = ax.bar(x+w/2, shared["new"],      w, color=ACCENT[3], alpha=0.85, label="New")
ax.set_xticks(x); ax.set_xticklabels(shared["metric"], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Score"); ax.set_title("Baseline vs New — Side by Side")
ax.legend(); ax.grid(True, axis="y", alpha=0.4)
ax.set_ylim(0, max(shared[["baseline","new"]].max()) * 1.15)

# Highlight regressions
for i, (_, row) in enumerate(shared.iterrows()):
    if row["regression"]:
        ax.axvspan(i-0.5, i+0.5, alpha=0.08, color=ACCENT[2])
        ax.text(i, ax.get_ylim()[1]*0.95, "⚠️", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("/tmp/regression.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()

print("\n💡 CI/CD Integration Pattern:")
print("""
# In your CI pipeline (e.g., GitHub Actions):
metrics = run_eval_suite(qa_pairs, retriever, generator, k=5)
comparison = compare_to_baseline(metrics, load_baseline('baselines/v1.0.json'))
regressions = comparison[comparison['regression']]
if len(regressions) > 0:
    print(regressions.to_string())
    sys.exit(1)  # Fail the build
""")


## 10. Evaluation Dashboard <a id='10-dashboard'></a>

A single-cell, comprehensive visual scorecard that brings together all metrics into one executive view.


In [ ]:
# ─── Master Evaluation Dashboard ────────────────────────────────────────────

fig = plt.figure(figsize=(20, 16))
fig.suptitle("🏆 RAG System Evaluation Dashboard", fontsize=18, fontweight="bold", y=0.98)
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.38)

# ── 1. Overall Scorecard (top-left large) ─────────────────────────────────────
ax_score = fig.add_subplot(gs[0, :2])
scorecard_metrics = {
    "Hit Rate@5"          : retrieval_df[retrieval_df["k"]==5]["hit_rate"].mean(),
    "NDCG@5"              : retrieval_df[retrieval_df["k"]==5]["ndcg"].mean(),
    "MRR"                 : retrieval_df[retrieval_df["k"]==5]["mrr"].mean(),
    "Context Recall"      : e2e_df["context_recall"].mean(),
    "Context Precision"   : e2e_df["context_precision"].mean(),
    "Answer Correctness"  : e2e_df["answer_correctness"].mean(),
    "ROUGE-1"             : gen_df["rouge1_f1"].mean(),
    "Groundedness"        : e2e_df["groundedness"].mean(),
}
labels_sc = list(scorecard_metrics.keys())
values_sc = list(scorecard_metrics.values())
colors_sc = [ACCENT[1] if v >= 0.7 else ACCENT[4] if v >= 0.5 else ACCENT[2] for v in values_sc]

bars = ax_score.barh(labels_sc[::-1], values_sc[::-1], color=colors_sc[::-1], alpha=0.88, height=0.65)
ax_score.set_xlim(0, 1.15)
ax_score.set_xlabel("Score")
ax_score.set_title("📊 Overall Metric Scorecard", fontsize=12, fontweight="bold")
ax_score.axvline(0.7, color="white", linestyle="--", alpha=0.4, label="Good threshold (0.7)")
ax_score.legend(fontsize=8)
ax_score.grid(True, axis="x", alpha=0.3)
for bar, val in zip(bars, values_sc[::-1]):
    grade = "A" if val>=0.8 else "B" if val>=0.7 else "C" if val>=0.5 else "D"
    ax_score.text(val+0.01, bar.get_y()+bar.get_height()/2,
                  f"{val:.2f}  [{grade}]", va="center", fontsize=9, color="white", fontweight="bold")

# ── 2. Radar Chart (top-right) ─────────────────────────────────────────────────
ax_radar = fig.add_subplot(gs[0, 2:], polar=True)
ax_radar.set_facecolor("#161b22")
radar_metrics = ["Hit Rate@5","NDCG@5","MRR","Context Recall","Answer Correct","Groundedness"]
radar_vals    = [scorecard_metrics.get("Hit Rate@5",0), scorecard_metrics.get("NDCG@5",0),
                 scorecard_metrics.get("MRR",0), scorecard_metrics.get("Context Recall",0),
                 scorecard_metrics.get("Answer Correctness",0), scorecard_metrics.get("Groundedness",0)]
angles = np.linspace(0,2*np.pi,len(radar_metrics),endpoint=False).tolist()
radar_vals_c = radar_vals + [radar_vals[0]]; angles_c = angles + [angles[0]]
ax_radar.plot(angles_c, radar_vals_c, "o-", color=ACCENT[0], linewidth=2)
ax_radar.fill(angles_c, radar_vals_c, alpha=0.2, color=ACCENT[0])
ax_radar.plot(angles_c, [0.8]*len(angles_c), "--", color=ACCENT[1], alpha=0.4, linewidth=1, label="Target (0.8)")
ax_radar.set_xticks(angles); ax_radar.set_xticklabels(radar_metrics, fontsize=8, color="#e6edf3")
ax_radar.set_ylim(0,1); ax_radar.set_title("System Radar", fontsize=11, color="#e6edf3", pad=15)
ax_radar.grid(color="#30363d"); ax_radar.legend(loc="upper right", fontsize=8)

# ── 3. Hallucination summary ──────────────────────────────────────────────────
ax_hall = fig.add_subplot(gs[1, 0])
if halluc_results:
    avg_halluc = np.mean([r["hallucination_rate"] for r in halluc_results])
    sev_counts = collections.Counter(r["severity"] for r in halluc_results)
    wedge_col  = [{"none":ACCENT[1],"minor":ACCENT[4],"moderate":"#ffa657","severe":ACCENT[2]}.get(s,ACCENT[0])
                  for s in sev_counts.keys()]
    ax_hall.pie(sev_counts.values(), labels=sev_counts.keys(), autopct="%1.0f%%",
                colors=wedge_col, textprops={"color":"white","fontsize":9})
    ax_hall.set_title(f"Hallucination Severity
(avg rate={avg_halluc:.0%})", fontsize=10)

# ── 4. LLM Judge scores ───────────────────────────────────────────────────────
ax_judge = fig.add_subplot(gs[1, 1])
judge_cols = ["faithfulness","relevance","completeness","clarity","overall"]
judge_means= [judge_df[c].mean() for c in judge_cols if c in judge_df.columns]
judge_lbls = [c.capitalize() for c in judge_cols if c in judge_df.columns]
bars_j = ax_judge.bar(range(len(judge_means)), judge_means,
                       color=[ACCENT[i%len(ACCENT)] for i in range(len(judge_means))], alpha=0.85)
ax_judge.set_xticks(range(len(judge_lbls))); ax_judge.set_xticklabels(judge_lbls, rotation=30, ha="right", fontsize=8)
ax_judge.set_ylim(0, 10.5); ax_judge.set_ylabel("Score /10")
ax_judge.set_title("LLM Judge Scores", fontsize=10); ax_judge.grid(True, axis="y", alpha=0.3)
ax_judge.axhline(7, color="white", linestyle="--", alpha=0.4, label="Good threshold")
ax_judge.legend(fontsize=8)
for bar, val in zip(bars_j, judge_means):
    ax_judge.text(bar.get_x()+bar.get_width()/2, val+0.1, f"{val:.1f}", ha="center", fontsize=8, color="white")

# ── 5. Latency breakdown ──────────────────────────────────────────────────────
ax_lat = fig.add_subplot(gs[1, 2])
if profile_traces:
    stage_avgs = {s: np.mean([t.get(s+"_ms",0) for t in profile_traces])
                  for s in ["embedding","retrieval","reranking","context_assembly","generation"]}
    total_avg  = sum(stage_avgs.values())
    pct_vals   = [v/total_avg*100 for v in stage_avgs.values()]
    ax_lat.pie(pct_vals, labels=[s.replace("_"," ").title() for s in stage_avgs.keys()],
               autopct="%1.0f%%",
               colors=[ACCENT[i%len(ACCENT)] for i in range(len(stage_avgs))],
               textprops={"color":"white","fontsize":8})
    ax_lat.set_title(f"Latency Breakdown
(avg total={total_avg:.0f}ms)", fontsize=10)

# ── 6. Cost per model ─────────────────────────────────────────────────────────
ax_cost = fig.add_subplot(gs[1, 3])
model_costs = {m.replace("claude-","").split("-2")[0]: estimate_pipeline_cost(10_000, llm_model=m)["per_month_usd"]
               for m in ["claude-haiku-4-5-20251001","claude-sonnet-4-20250514","claude-opus-4-6"]}
bars_c = ax_cost.bar(model_costs.keys(), model_costs.values(),
                      color=[ACCENT[1],ACCENT[0],ACCENT[2]], alpha=0.85, width=0.5)
ax_cost.set_ylabel("Monthly Cost ($)")
ax_cost.set_title("Monthly Cost by Model
(10K q/day)", fontsize=10)
ax_cost.grid(True, axis="y", alpha=0.3)
for bar, val in zip(bars_c, model_costs.values()):
    ax_cost.text(bar.get_x()+bar.get_width()/2, val+max(model_costs.values())*0.02,
                 f"${val:,.0f}", ha="center", fontsize=9, color="white")

# ── 7. Regression status ──────────────────────────────────────────────────────
ax_reg = fig.add_subplot(gs[2, :2])
n_reg    = comparison_df["regression"].sum() if "comparison_df" in dir() else 0
n_imp    = (comparison_df["delta"] > 0).sum() if "comparison_df" in dir() else 0
n_stable = len(comparison_df) - n_reg - n_imp if "comparison_df" in dir() else 0
ax_reg.barh(["Regressions","Improved","Stable"],[n_reg,n_imp,n_stable],
            color=[ACCENT[2],ACCENT[1],ACCENT[0]], alpha=0.85, height=0.5)
ax_reg.set_xlabel("Count"); ax_reg.set_title("Regression Test Status (Baseline vs K=3)", fontsize=10)
ax_reg.grid(True, axis="x", alpha=0.4)
status_txt = "🔴 FAIL — regressions detected" if n_reg > 0 else "✅ PASS — no regressions"
ax_reg.text(0.98, 0.92, status_txt, transform=ax_reg.transAxes,
            ha="right", fontsize=10, color=ACCENT[2] if n_reg > 0 else ACCENT[1])

# ── 8. Metric trend (simulated versions) ─────────────────────────────────────
ax_trend = fig.add_subplot(gs[2, 2:])
versions = ["v0.1","v0.2","v0.3","v0.4","v1.0"]
# Simulated improvement trend
ndcg_trend  = [0.42, 0.51, 0.58, 0.63, retrieval_df[retrieval_df["k"]==5]["ndcg"].mean()]
rouge_trend = [0.28, 0.33, 0.38, 0.41, gen_df["rouge1_f1"].mean()]
grnd_trend  = [0.55, 0.62, 0.68, 0.72, e2e_df["groundedness"].mean()]
ax_trend.plot(versions, ndcg_trend,  "o-", color=ACCENT[0], label="NDCG@5",  linewidth=2)
ax_trend.plot(versions, rouge_trend, "s-", color=ACCENT[3], label="ROUGE-1", linewidth=2)
ax_trend.plot(versions, grnd_trend,  "^-", color=ACCENT[1], label="Groundedness", linewidth=2)
ax_trend.set_ylabel("Score"); ax_trend.set_title("Metric Trend Across Pipeline Versions", fontsize=10)
ax_trend.legend(fontsize=8); ax_trend.grid(True, alpha=0.4); ax_trend.set_ylim(0,1.05)
ax_trend.fill_between(range(len(versions)), 0.7, 1.0, alpha=0.05, color=ACCENT[1], label="Target zone")

plt.savefig("/tmp/dashboard.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ Dashboard rendered.")


## 11. 🚀 Production Harness <a id='11-harness'></a>

**Drop this into any RAG project.** Implement the three interfaces below and you get all 20+ metrics automatically.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  🚀  PRODUCTION RAG EVALUATION HARNESS
# ─────────────────────────────────────────────────────────────────────────────
#  HOW TO USE:
#  1. Implement YOUR_RETRIEVER — must have a .retrieve(query, k) method
#     returning List[{"id": str, "text": str, "score": float}]
#  2. Implement YOUR_GENERATOR — must have a .generate(query, docs) method
#     returning {"answer": str, "in_tok": int, "out_tok": int}
#  3. Provide YOUR_QA_PAIRS — list of ground-truth dicts (see format below)
#  4. Run evaluate_rag_system() — get the full report
# ═════════════════════════════════════════════════════════════════════════════

# ── Format for YOUR_QA_PAIRS ──────────────────────────────────────────────────
QA_PAIR_SCHEMA = {
    "query"           : "string — the user question",
    "relevant_doc_ids": ["list","of","doc","ids"],   # IDs of truly relevant docs
    "answer_gt"       : "string — the correct answer (reference)",
    "topic"           : "optional string — category for grouping",
}

# ── Drop-in Interface ──────────────────────────────────────────────────────────
class RAGEvaluationHarness:
    """
    Production-ready RAG evaluation harness.
    Plug in your retriever and generator — get all metrics out.
    
    Usage:
        harness = RAGEvaluationHarness(
            retriever    = YOUR_RETRIEVER,
            generator    = YOUR_GENERATOR,
            qa_pairs     = YOUR_QA_PAIRS,
            k            = 5,
            llm_judge    = True,   # Set False to skip API calls
            hallucination= True,   # Set False to skip API calls
        )
        report = harness.evaluate()
        harness.print_report(report)
        harness.save_report(report, "eval_run_v1.json")
    """

    def __init__(self, retriever, generator, qa_pairs: List[Dict],
                 k: int = 5, llm_judge: bool = True,
                 hallucination: bool = True,
                 baseline: Optional[RAGBaseline] = None):
        self.retriever     = retriever
        self.generator     = generator
        self.qa_pairs      = qa_pairs
        self.k             = k
        self.llm_judge     = llm_judge
        self.hallucination = hallucination
        self.baseline      = baseline

    def evaluate(self) -> Dict:
        """Run the full evaluation suite. Returns a comprehensive results dict."""
        results = {
            "timestamp"    : time.strftime("%Y-%m-%d %H:%M:%S"),
            "n_queries"    : len(self.qa_pairs),
            "k"            : self.k,
            "retrieval"    : {},
            "generation"   : {},
            "e2e"          : {},
            "llm_judge"    : {},
            "hallucination": {},
            "latency"      : {},
            "cost"         : {},
            "per_query"    : [],
        }

        gen_rows, e2e_rows, judge_rows, hall_rows = [], [], [], []
        latency_gen_ms = []

        print(f"\n🚀 RAG Evaluation Harness — evaluating {len(self.qa_pairs)} queries at K={self.k}")
        print("="*65)

        for i, qa in enumerate(self.qa_pairs):
            q_result = {"query": qa["query"], "topic": qa.get("topic","")}

            # ── Retrieve ──────────────────────────────────────────────────────
            t_ret  = time.perf_counter()
            r_docs = self.retriever.retrieve(qa["query"], self.k)
            t_ret  = (time.perf_counter()-t_ret)*1000
            r_ids  = [d["id"] for d in r_docs]
            ctx    = " ".join(d["text"] for d in r_docs)

            # ── Retrieval metrics ──────────────────────────────────────────────
            q_result["hit_rate"]  = hit_rate_at_k(r_ids, qa["relevant_doc_ids"], self.k)
            q_result["precision"] = precision_at_k(r_ids, qa["relevant_doc_ids"], self.k)
            q_result["recall"]    = recall_at_k(r_ids, qa["relevant_doc_ids"], self.k)
            q_result["mrr"]       = mean_reciprocal_rank(r_ids, qa["relevant_doc_ids"])
            q_result["ndcg"]      = ndcg_at_k(r_ids, qa["relevant_doc_ids"], self.k)
            q_result["context_recall"]   = context_recall(r_ids, qa["relevant_doc_ids"])
            q_result["context_precision"]= context_precision(r_ids, qa["relevant_doc_ids"])

            # ── Generate ──────────────────────────────────────────────────────
            t_gen = time.perf_counter()
            gen   = self.generator.generate(qa["query"], r_docs[:3], max_tokens=200)
            t_gen = (time.perf_counter()-t_gen)*1000
            hyp   = gen["answer"]
            latency_gen_ms.append(t_gen)

            # ── Generation metrics ────────────────────────────────────────────
            q_result["rouge1"]           = rouge_n(hyp, qa["answer_gt"], 1)["f1"]
            q_result["rouge2"]           = rouge_n(hyp, qa["answer_gt"], 2)["f1"]
            q_result["rougeL"]           = rouge_l(hyp, qa["answer_gt"])["f1"]
            q_result["token_f1"]         = token_f1(hyp, qa["answer_gt"])
            q_result["semantic_sim"]     = semantic_similarity(hyp, qa["answer_gt"])
            q_result["answer_correctness"]= answer_correctness(hyp, qa["answer_gt"])
            q_result["groundedness"]     = groundedness_score(hyp, ctx)
            q_result["answer_completeness"]= answer_completeness(hyp, qa["answer_gt"])
            q_result["in_tok"]           = gen["in_tok"]
            q_result["out_tok"]          = gen["out_tok"]
            q_result["gen_latency_ms"]   = t_gen
            q_result["hypothesis"]       = hyp[:150]

            # ── LLM Judge (optional) ──────────────────────────────────────────
            if self.llm_judge:
                j = llm_judge_single(qa["query"], hyp, ctx[:1000])
                q_result["judge_faithfulness"] = j.get("faithfulness",5)
                q_result["judge_relevance"]    = j.get("relevance",5)
                q_result["judge_overall"]      = j.get("overall",5)

            # ── Hallucination (optional) ──────────────────────────────────────
            if self.hallucination:
                h = hallucination_score(hyp, ctx)
                q_result["hallucination_rate"] = h["hallucination_rate"]
                q_result["hallucination_severity"] = h["severity"]

            results["per_query"].append(q_result)
            status = "✅" if q_result["hit_rate"] >= 0.8 else "⚠️"
            print(f"  {status} Q{i+1} | hit={q_result['hit_rate']:.2f} "
                  f"ndcg={q_result['ndcg']:.2f} "
                  f"R1={q_result['rouge1']:.2f} "
                  f"grnd={q_result['groundedness']:.2f}")

        # ── Aggregate ─────────────────────────────────────────────────────────
        pq = pd.DataFrame(results["per_query"])

        ret_cols  = ["hit_rate","precision","recall","mrr","ndcg","context_recall","context_precision"]
        gen_cols  = ["rouge1","rouge2","rougeL","token_f1","semantic_sim","answer_correctness","groundedness"]
        judge_cols= [c for c in ["judge_faithfulness","judge_relevance","judge_overall"] if c in pq.columns]
        hall_cols = [c for c in ["hallucination_rate"] if c in pq.columns]

        results["retrieval"]     = pq[ret_cols].mean().round(4).to_dict()
        results["generation"]    = pq[gen_cols].mean().round(4).to_dict()
        results["llm_judge"]     = pq[judge_cols].mean().round(4).to_dict() if judge_cols else {}
        results["hallucination"] = pq[hall_cols].mean().round(4).to_dict() if hall_cols else {}
        results["latency"]       = {
            "gen_mean_ms": round(np.mean(latency_gen_ms),1),
            "gen_p95_ms" : round(np.percentile(latency_gen_ms,95),1),
        }
        results["cost"] = estimate_pipeline_cost(
            10_000, llm_model=ACTIVE_MODEL, k_retrieved=self.k)["per_query"]

        # ── Regression check ──────────────────────────────────────────────────
        if self.baseline:
            flat_metrics = {**results["retrieval"], **results["generation"]}
            results["regression"] = compare_to_baseline(flat_metrics, self.baseline).to_dict("records")

        return results

    def print_report(self, results: Dict):
        """Print a formatted evaluation report."""
        print("\n" + "═"*70)
        print("  📊 RAG EVALUATION REPORT")
        print("═"*70)
        print(f"  Queries evaluated : {results['n_queries']}")
        print(f"  K retrieved       : {results['k']}")
        print(f"  Timestamp         : {results['timestamp']}")

        print("\n  RETRIEVAL METRICS")
        for k,v in results["retrieval"].items():
            grade = "🟢" if v>=0.7 else "🟡" if v>=0.5 else "🔴"
            print(f"    {grade} {k:<30} {v:.4f}")

        print("\n  GENERATION METRICS")
        for k,v in results["generation"].items():
            grade = "🟢" if v>=0.5 else "🟡" if v>=0.3 else "🔴"
            print(f"    {grade} {k:<30} {v:.4f}")

        if results.get("llm_judge"):
            print("\n  LLM JUDGE SCORES (out of 10)")
            for k,v in results["llm_judge"].items():
                grade = "🟢" if v>=7 else "🟡" if v>=5 else "🔴"
                print(f"    {grade} {k:<30} {v:.2f}/10")

        if results.get("hallucination"):
            rate = results["hallucination"].get("hallucination_rate",0)
            sev  = "🟢 Low" if rate < 0.1 else "🟡 Medium" if rate < 0.3 else "🔴 High"
            print(f"\n  HALLUCINATION: {sev} ({rate:.0%} avg rate)")

        print(f"\n  COST: ${results['cost'].get('total_usd',0):.6f} per query")
        print("═"*70)

    def save_report(self, results: Dict, path: str):
        with open(path, "w") as f:
            # Convert numpy floats for JSON serialization
            json.dump(results, f, indent=2, default=lambda x: float(x) if hasattr(x,"item") else str(x))
        print(f"  💾 Report saved to {path}")


# ── Run the harness on our test pipeline ─────────────────────────────────────
harness = RAGEvaluationHarness(
    retriever    = RETRIEVER,
    generator    = GENERATOR,
    qa_pairs     = GROUND_TRUTH_QA[:5],   # Limit for demo (5 queries)
    k            = 5,
    llm_judge    = True,    # ← set False to skip LLM judge API calls
    hallucination= True,    # ← set False to skip hallucination API calls
    baseline     = BASELINE,
)

harness_results = harness.evaluate()
harness.print_report(harness_results)
harness.save_report(harness_results, "/tmp/rag_eval_report.json")
TRACKER.summary()


## Summary — Metric Selection Guide

### Which metrics to use in production

| Stage | Must Have | Nice to Have | Skip When |
|-------|-----------|-------------|-----------|
| **Retrieval** | NDCG@K, Hit Rate@K | MAP@K, MRR | Latency budget is tight |
| **Generation** | Token F1, Semantic Sim | ROUGE-1/2/L | No reference answers available |
| **E2E** | Context Recall, Faithfulness | Context Precision | Low-stakes applications |
| **Quality Gate** | LLM Judge (G-Eval) | Pairwise comparison | High query volume ($$$) |
| **Safety** | Hallucination Rate | Claim verification | Trusted, closed-domain corpus |
| **Ops** | P95 Latency, Cost/Query | Per-stage profiling | Prototype phase |
| **CI/CD** | Regression vs baseline | Trend tracking | One-off scripts |

### Recommended eval cadence

```
Every PR        → Offline metrics (Retrieval + ROUGE + E2E) — fast, free
Every release   → + LLM Judge on 50-100 queries
Every quarter   → + Hallucination audit on random 200 queries  
On model change → Full eval suite + regression report
On data change  → Context Recall + Faithfulness focus
```

### Production alert thresholds (starting points)

```python
ALERTS = {
    "ndcg@5"          : 0.65,   # below → retrieval regression
    "context_recall"  : 0.70,   # below → missing relevant context
    "faithfulness"    : 0.75,   # below → hallucination risk
    "hallucination_rate": 0.15, # above → trigger human review
    "p95_latency_ms"  : 3000,   # above → SLA breach
    "cost_per_query"  : 0.005,  # above → cost optimization needed
}
```

---
*RAG Evaluation Master Notebook — drop `RAGEvaluationHarness` into any production RAG pipeline.*
